<a href="https://colab.research.google.com/github/awinarko-hue/Project_hue/blob/main/Legal/8b_GraphRAG_Exp_03_b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GRAPH-RAG KOMDIGI — Exp_03_b (perbaikan dari Exp_03_a)

Pipeline terpadu: korpus regulasi → ekstraksi rujukan & norma deontik → Knowledge Graph → retrieval
(BM25, Dense, Hybrid, Graph-RAG, varian sadar-deontik) → evaluasi retrieval & generasi → audit akurasi rujukan.

**Perubahan utama terhadap Exp_03_a** (kode temuan mengacu pada hasil review):

| Temuan | Perbaikan di 03_b | Sel |
|---|---|---|
| A1 presisi rujukan internal 23% (dugaan: omnibus & lampiran) | Diagnosis struktur per dokumen; ekstraksi **v3** sadar-omnibus (rujukan di blok "Ketentuan … UU X diubah" diarahkan ke UU X); rujukan dari pasal-hub/lampiran ditandai & dikeluarkan dari KG; analisis label audit v2 per dokumen dan terhadap perubahan v3 | §1a, §1b, §8a |
| A2 hasil 20 kueri vs 46 kueri bertentangan | Subset 20 kueri dihapus; satu evaluasi pada seluruh gold tervalidasi + CI bootstrap | §7 |
| A3 ablasi `menyebut`/`merujuk_dokumen` tak berpengaruh | Kuota kandidat **per jenis sisi**; entitas generik dibuang; `merujuk_dokumen` ke pasal definisi (bukan P1 buta) | §4, §5 |
| A4 saudara satu pasal dipilih menurut urutan sisipan | Kandidat diskor `bobot_relasi × cosine(kueri, kandidat)` | §5 |
| B1 effect size Wilcoxon salah tanda | r rank-biserial dari W+ dan W− + koreksi Holm | §7 |
| B2/B3 checkpoint basi & jawaban gagal tersimpan | Kunci checkpoint = hash(sistem, pertanyaan, konfigurasi); jawaban gagal tidak disimpan; judge diulang terpisah | §7b |
| B4 Citation F1 hanya nomor pasal | Citation F1 **ketat** (dokumen + pasal) + versi longgar untuk pembanding | §7b |
| B5 judge = generator, konteks judge terpotong | Penyedia judge terpisah (default Gemini); konteks judge = konteks generator | §6, §7b |
| B6 cache embedding hanya cek jumlah baris | Kunci cache = hash(model + teks) | §2 |
| B7 koreksi metadata PP51 tak pernah jalan | Koreksi generik `METADATA_FIX` + cek konsistensi doc_id vs nomor/tahun | §1 |
| B8 reachability salah label | Jangkauan dihitung per jenis relasi | §7 |
| B9 P@k tingkat pasal tidak adil | Ambil `K_POOL` chunk, dedup ke pasal, baru potong k | §7 |
| Label `asal` 03_a menghitung hybrid peringkat 7–12 sebagai "graf" | `asal` = seed / hybrid / graf + kolom `rel_graf` | §5 |
| Filter metadata dilewati oleh ekspansi graf | Hasil ekspansi ikut disaring | §5 |
| Duplikat Perppu 2/2022 ↔ UU 6/2023 | Dedup kembar + peta remap gold | §1, §7 |
| `chunk_id` tidak stabil | Snapshot `chunk_id`+hash isi; peringatan bila bergeser | §1 |
| Deontik tak dipakai retrieval (G1) | Deteksi intent kueri + bonus deontik → konfigurasi 5 & 6 untuk diuji | §1c, §5, §7 |
| Uji | Sel self-test (§9) | §9 |

> Jalankan berurutan dari atas. Konfigurasi hanya diubah di §0.

## §0 — Konfigurasi & Lingkungan (ubah hanya di sel ini)

In [ ]:
import os, re, sys, glob, json, time, math, pickle, random, hashlib, platform, warnings
from functools import lru_cache
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import networkx as nx

IN_COLAB = "google.colab" in sys.modules
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)


def _env(nama, default=None):
    return os.environ.get(nama, default)


def _env_bool(nama, default):
    v = os.environ.get(nama)
    return default if v is None else v.strip().lower() in {"1", "true", "ya", "yes", "y"}


def pip_install(paket: str):
    """Instalasi dependensi. Di Colab otomatis; di lokal jalankan sekali di terminal:
    pip install sentence-transformers faiss-cpu rank_bm25 networkx pyvis openpyxl scipy scikit-learn"""
    if IN_COLAB:
        os.system(f"{sys.executable} -m pip install -q {paket}")


try:
    import faiss  # noqa
    from sentence_transformers import SentenceTransformer  # noqa
    from rank_bm25 import BM25Okapi  # noqa
except Exception:
    pip_install("sentence-transformers faiss-cpu rank_bm25==0.2.2 pyvis==0.3.2 openpyxl")
    import faiss  # noqa
    from sentence_transformers import SentenceTransformer  # noqa
    from rank_bm25 import BM25Okapi  # noqa

# ---------------------------------------------------------------------------
# PATH (variabel lingkungan KORPUS_DIR / PROJECT7_DIR dapat menimpa nilai default)
# ---------------------------------------------------------------------------
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Gagal mount Google Drive: {e}")
    _BASE_DEFAULT = "/content/drive/MyDrive/Tesis_SDPPI/Project 7/GraphRAG_Komdigi"
    _KORPUS_DEFAULT = "/content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Korpus Komdigi"
else:
    _BASE_DEFAULT = "Project 7"
    _KORPUS_DEFAULT = None

BASE_DIR = Path(_env("PROJECT7_DIR", _BASE_DEFAULT))
KORPUS_DIR = Path(_env("KORPUS_DIR", _KORPUS_DEFAULT or str(BASE_DIR / "korpus")))
IFACT_DIR = BASE_DIR / "artifacts_03b"    # cache 03_b (terpisah dari cache 03_a)
HASIL_DIR = BASE_DIR / "hasil_03b"        # tabel hasil untuk Bab 4
AUDIT_DIR = BASE_DIR / "audit_rujukan"    # dipakai bersama 03_a (lembar v2 dibaca, v3 dibuat)
for d in (IFACT_DIR, HASIL_DIR, AUDIT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# KORPUS
# ---------------------------------------------------------------------------
# Pasangan dokumen kembar {dokumen_dibuang: dokumen_disimpan}. Baris dokumen_dibuang yang
# isinya identik dengan baris dokumen_disimpan dibuang; chunk_id-nya dipetakan (remap gold).
DOK_KEMBAR = {"Perppu_2_2022": "UU_6_2023"}
# Dokumen omnibus/pengubah: rujukan "Pasal N" di dalam blok "Beberapa ketentuan dalam UU X diubah"
# diarahkan ke UU X (ekstraksi v3).
DOK_OMNIBUS = {"UU_6_2023", "Perppu_2_2022"}
# Pasal dengan satuan sebanyak ini dianggap hub/lampiran (contoh 03_a: Perdirjen_07_2021::P75 = 702)
PASAL_HUB_MIN_SATUAN = 150
SNAPSHOT_PERBARUI = False   # True hanya bila gold sudah disesuaikan dengan korpus baru

# ---------------------------------------------------------------------------
# EKSTRAKSI & KG
# ---------------------------------------------------------------------------
EKSTRAKSI_AKTIF = "v3"      # 'v3' (sadar-omnibus, 03_b) | 'v2' (03_a) | 'v1' (baseline)
DEONTIK_AKTIF = "v2"        # 'v2' (pola diperketat, 03_b) | 'v1' (03_a)
KG_ABAIKAN_OMNIBUS_TANPA_KONTEKS = True
KG_ABAIKAN_RUJUKAN_DARI_HUB = True
KG_DOK_TANPA_RUJUKAN_INTERNAL = set()   # isi setelah membaca diagnosis §8a, mis. {"Perdirjen_07_2021"}
ENT_GENERIK_MAKS_PROPORSI = 0.01        # entitas yang muncul di >1% satuan dianggap generik

# ---------------------------------------------------------------------------
# RETRIEVAL
# ---------------------------------------------------------------------------
EMBED_MODEL_NAME = _env("EXP_EMBED_MODEL", "paraphrase-multilingual-MiniLM-L12-v2")
# Alternatif: "intfloat/multilingual-e5-base" (awalan query:/passage: ditambahkan otomatis)
TEKS_EMBED_SERTAKAN_TIPE = True   # sama dengan 03_a; set False untuk menghindari label heuristik di embedding
RRF_K = 60
K_SEED = 6                  # seed untuk ekspansi graf
N_POOL_HYBRID = 12          # kandidat hybrid yang ikut fusi Graph-RAG (independen dari k)
ALPHA_GRAF = 1.2
DECAY = 0.7
BOBOT_SISI = {"memuat": 1.0, "merujuk": 0.85, "merujuk_dokumen": 0.65, "menyebut": 0.5}
KUOTA_SISI = {"memuat": 3, "merujuk": 3, "merujuk_dokumen": 1, "menyebut": 1}
BETA_DEONTIK = 0.5          # ditetapkan a priori (JANGAN di-tuning pada gold uji)

# ---------------------------------------------------------------------------
# EVALUASI
# ---------------------------------------------------------------------------
GOLD_PATH = Path(_env("EXP_GOLD_PATH", str(BASE_DIR / "gold_standard.csv")))
K_LIST = (3, 5, 10)
K_POOL = 30                 # kandidat yang diambil sebelum dedup ke tingkat pasal
BOOTSTRAP_B = 2000

# ---------------------------------------------------------------------------
# LLM (generator & judge DIPISAH untuk mengurangi bias menilai diri sendiri)
# ---------------------------------------------------------------------------
USE_LLM = _env_bool("EXP_USE_LLM", True)
LLM_PROVIDER = _env("EXP_LLM_PROVIDER", "groq")       # groq | gemini | mock (uji tanpa API)
GROQ_MODEL = "openai/gpt-oss-120b"
GEMINI_MODEL = "gemini-3.5-flash-lite"
JUDGE_PROVIDER = _env("EXP_JUDGE_PROVIDER", "gemini")  # groq | gemini | mock
JUDGE_MODEL = None          # None → model default penyedia judge
KONTEKS_MAX_CHAR = 12000    # konteks yang SAMA dipakai generator dan judge
JEDA_API_DETIK = 2.5

RUN_EVAL_RETRIEVAL = True
RUN_EVAL_GENERASI = _env_bool("EXP_RUN_GEN", True)
SISTEM_EVAL_GENERASI = ["3. Hybrid (RRF)", "4. Graph-RAG"]
RUN_AUDIT = True
RUN_SELFTEST = True

VERSI_KODE = "exp03b-1.0"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")

print("Lingkungan   :", "Colab" if IN_COLAB else "Lokal")
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | networkx {nx.__version__}")
print("Korpus dicari:", KORPUS_DIR)
_csv = sorted(os.path.basename(f) for f in glob.glob(str(KORPUS_DIR / "korpus*.csv")))
print(f"File terdeteksi ({len(_csv)}):", _csv)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Lingkungan   : Colab
python 3.13.18 | pandas 2.2.3 | numpy 2.4.6 | networkx 3.7
Korpus dicari: /content/drive/MyDrive/Tesis_SDPPI/Project 7/Aplikasi_Skema/Korpus Komdigi
File terdeteksi (23): ['korpusPM172018.csv', 'korpusPM42025.csv', 'korpusPM52021.csv', 'korpusPM82024.csv', 'korpusPM82026.csv', 'korpusPM92023.csv', 'korpusPM_2_2023.csv', 'korpusPM_2_2025.csv', 'korpusPP282025.csv', 'korpusPP462021.csv', 'korpusPP492005.csv', 'korpusPP512005.csv', 'korpusPP522000.csv', 'korpusPP532000.csv', 'korpusPerdirjen32024.csv', 'korpusPerdirjen72021.csv', 'korpusPerppu22022.csv', 'korpusPerpres_174_2024.csv', 'korpusUU122011.csv', 'korpusUU322002.csv', 'korpusUU361999.csv', 'korpusUU382009.csv', 'korpusUU62023.csv']


## §1 — Pemuatan, Pembersihan, Dedup Kembar & Snapshot `chunk_id`
- Koreksi metadata generik (`METADATA_FIX`) + cek konsistensi `doc_id` ↔ kolom nomor/tahun (B7).
- `chunk_id` memakai skema yang sama dengan 03_a agar gold lama tetap berlaku; snapshot disimpan
  untuk mendeteksi pergeseran ID saat korpus berubah.
- Baris kembar Perppu 2/2022 ↔ UU 6/2023 dibuang dari dokumen yang tidak dipilih; peta kembar dipakai
  untuk remap gold (§7) sehingga sistem tidak dihukum karena mengambil "kembaran".

In [ ]:
SCHEMA = ["doc_id", "nama_dokumen", "jenis", "tahun", "nomor", "bab", "nama_bab",
          "bagian", "paragraf", "pasal", "ayat", "huruf", "angka", "tipe", "isi",
          "referensi", "kata_kunci", "entitas"]
VALID_TIPE = {"Ketentuan", "Kewajiban", "Sanksi", "Definisi", "Hak", "Larangan"}

# Koreksi metadata per file. Catatan: 03_a menyasar 'korpusPP512002.csv' yang TIDAK ADA di folder,
# sehingga koreksi itu tidak pernah berjalan. Isi di sini bila cek konsistensi di bawah menandai masalah.
METADATA_FIX = {
    # "korpusPP512005.csv": {"doc_id": "PP_51_2005", "tahun": 2005, "nomor": 51},
}

KODE_DOCID = {"UU": "UU", "PP": "PP", "PERPPU": "PERPPU", "PERPRES": "PERPRES",
              "PERMEN": "PERMEN", "PERDIRJEN": "PERDIRJEN", "KEPMEN": "KEPMEN"}


def _s(x) -> str:
    """str() yang aman terhadap NaN/pd.NA (03_a menghasilkan '<NA>' / 'nan')."""
    return "" if x is None or (not isinstance(x, (list, tuple, set)) and pd.isna(x)) else str(x)


def _md5(teks: str, n: int = 12) -> str:
    return hashlib.md5(teks.encode("utf-8")).hexdigest()[:n]


def parse_docid(doc_id):
    m = re.match(r"([A-Za-z]+)_0*(\d+)_(\d{4})", str(doc_id))
    return None if not m else (KODE_DOCID.get(m.group(1).upper(), m.group(1).upper()),
                               int(m.group(2)), int(m.group(3)))


def load_korpus(korpus_dir: Path) -> pd.DataFrame:
    frames, laporan = [], []
    for path in sorted(glob.glob(str(Path(korpus_dir) / "korpus*.csv"))):
        df = pd.read_csv(path)
        hilang = [c for c in SCHEMA if c not in df.columns]
        if hilang:
            print(f"[LEWATI] {os.path.basename(path)} — kolom hilang: {hilang}")
            continue
        df = df[SCHEMA].copy()
        df["sumber_file"] = os.path.basename(path)
        frames.append(df)
        laporan.append((os.path.basename(path), len(df), df.doc_id.iloc[0]))
    if not frames:
        raise FileNotFoundError(f"Tidak ada korpus*.csv valid di {korpus_dir}")
    korpus = pd.concat(frames, ignore_index=True)
    print(pd.DataFrame(laporan, columns=["file", "baris", "doc_id"]).to_string(index=False))
    return korpus


def cek_konsistensi_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """Bandingkan nomor/tahun yang tersirat di doc_id dengan kolom nomor/tahun dan nama file."""
    out = []
    for (fname, did), g in df.groupby(["sumber_file", "doc_id"]):
        k = parse_docid(did)
        nomor = pd.to_numeric(g["nomor"], errors="coerce").mode()
        tahun = pd.to_numeric(g["tahun"], errors="coerce").mode()
        nomor = int(nomor.iloc[0]) if len(nomor) else None
        tahun = int(tahun.iloc[0]) if len(tahun) else None
        angka_file = "".join(re.findall(r"\d", fname))
        masalah = []
        if k is None:
            masalah.append("doc_id tak terurai")
        else:
            if nomor is not None and nomor != k[1]:
                masalah.append(f"nomor kolom={nomor} ≠ doc_id={k[1]}")
            if tahun is not None and tahun != k[2]:
                masalah.append(f"tahun kolom={tahun} ≠ doc_id={k[2]}")
            if str(k[2]) not in angka_file:
                masalah.append(f"tahun doc_id {k[2]} tidak ada di nama file")
        out.append({"file": fname, "doc_id": did, "nomor": nomor, "tahun": tahun,
                    "masalah": "; ".join(masalah) or "-"})
    return pd.DataFrame(out)


def bersihkan(korpus: pd.DataFrame) -> pd.DataFrame:
    df = korpus.copy()
    # 1. Koreksi metadata generik (SEBELUM membentuk chunk_id)
    for fname, fix in METADATA_FIX.items():
        m = df["sumber_file"] == fname
        if not m.any():
            print(f"[PERINGATAN] METADATA_FIX: file {fname} tidak ditemukan — koreksi tidak diterapkan.")
            continue
        for kol, nilai in fix.items():
            df.loc[m, kol] = nilai
        print(f"METADATA_FIX diterapkan pada {fname}: {fix}")

    # 2. Saring baris rusak (baris tabel teknis yang mengotori kolom 'tipe')
    kotor = ~df["tipe"].isin(VALID_TIPE)
    if kotor.sum():
        print(f"Membuang {int(kotor.sum())} baris anomali tipe.")
        df = df[~kotor].copy()

    # 3. Normalisasi teks isi
    df["isi"] = df["isi"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = df[df["isi"].str.len() > 0].copy()

    # 4. Standarisasi metadata
    df["jenis"] = df["jenis"].astype(str).str.strip()
    df["tahun"] = pd.to_numeric(df["tahun"], errors="coerce").astype("Int64")
    for c in ["pasal", "ayat", "angka"]:
        df[c] = df[c].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    df["huruf"] = df["huruf"].astype("string").str.strip().str.lower()

    # 5. Buang duplikat persis di dalam dokumen yang sama
    n0 = len(df)
    df = df.drop_duplicates(subset=["doc_id", "pasal", "ayat", "huruf", "angka", "isi"])
    if n0 - len(df):
        print(f"Duplikat dalam-dokumen dibuang: {n0 - len(df)}")

    # 6. chunk_id hierarkis — skema IDENTIK dengan 03_a (kompatibel dengan gold lama)
    def buat_id(r):
        parts = [str(r["doc_id"]), f"P{r['pasal']}" if pd.notna(r["pasal"]) else "P-"]
        if pd.notna(r["ayat"]):  parts.append(f"a{r['ayat']}")
        if pd.notna(r["huruf"]): parts.append(f"h{r['huruf']}")
        if pd.notna(r["angka"]): parts.append(f"n{r['angka']}")
        return "__".join(parts)

    df["chunk_id"] = df.apply(buat_id, axis=1)
    dup = df.duplicated("chunk_id", keep=False)
    df.loc[dup, "chunk_id"] = (df.loc[dup, "chunk_id"] + "__" +
                               df.loc[dup].groupby("chunk_id").cumcount().astype(str))
    return df.reset_index(drop=True)


def cek_snapshot_chunk_id(df: pd.DataFrame, path: Path):
    """Deteksi pergeseran chunk_id (mis. akhiran __0/__1 berubah karena urutan baris)."""
    kini = pd.DataFrame({"chunk_id": df["chunk_id"], "hash_isi": df["isi"].map(lambda t: _md5(t, 10))})
    if not path.exists() or SNAPSHOT_PERBARUI:
        kini.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"Snapshot chunk_id disimpan ({len(kini)} ID) → {path.name}")
        return
    lama = pd.read_csv(path)
    gab = lama.merge(kini, on="chunk_id", how="left", suffixes=("_lama", "_kini"))
    hilang = gab["hash_isi_kini"].isna().sum()
    berubah = ((gab["hash_isi_kini"].notna()) & (gab["hash_isi_lama"] != gab["hash_isi_kini"])).sum()
    baru = len(set(kini["chunk_id"]) - set(lama["chunk_id"]))
    if hilang or berubah:
        print(f"[PERINGATAN] chunk_id bergeser vs snapshot: {hilang} hilang, {berubah} isi berubah, "
              f"{baru} baru. Periksa gold sebelum evaluasi (lihat {path.name}_diff.csv).")
        gab[gab["hash_isi_kini"].isna() | (gab["hash_isi_lama"] != gab["hash_isi_kini"])] \
            .to_csv(path.with_name(path.stem + "_diff.csv"), index=False, encoding="utf-8-sig")
    else:
        print(f"Snapshot chunk_id cocok ({len(lama)} ID; {baru} ID baru).")


def dedup_kembar(df: pd.DataFrame, dok_kembar: dict):
    """Buang baris dok_dibuang yang isinya identik dengan dok_disimpan. Return (korpus, peta)."""
    peta, buang_idx, laporan = {}, [], []
    docs = set(df["doc_id"])
    for buang, simpan in dok_kembar.items():
        if buang not in docs or simpan not in docs:
            print(f"Dedup kembar {buang}→{simpan} dilewati (dokumen tidak ada di korpus).")
            continue
        kunci = lambda t: re.sub(r"\W+", " ", str(t).lower()).strip()  # noqa: E731
        indeks_simpan = {}
        for r in df[df["doc_id"] == simpan].itertuples():
            indeks_simpan.setdefault(kunci(r.isi), r.chunk_id)
        g = df[df["doc_id"] == buang]
        n_kembar = 0
        for r in g.itertuples():
            tujuan = indeks_simpan.get(kunci(r.isi))
            if tujuan is not None:
                peta[r.chunk_id] = tujuan
                buang_idx.append(r.Index)
                n_kembar += 1
        laporan.append({"dibuang": buang, "disimpan": simpan, "baris_dibuang": len(g),
                        "kembar_dihapus": n_kembar, "unik_dipertahankan": len(g) - n_kembar})
    if laporan:
        print(pd.DataFrame(laporan).to_string(index=False))
    return df.drop(index=buang_idx).reset_index(drop=True), peta


korpus_mentah = load_korpus(KORPUS_DIR)
_meta = cek_konsistensi_metadata(korpus_mentah)
if (_meta["masalah"] != "-").any():
    print("\n[PERINGATAN] Inkonsistensi metadata:")
    print(_meta[_meta["masalah"] != "-"].to_string(index=False))
else:
    print("\nCek konsistensi metadata: OK untuk semua dokumen.")

korpus_full = bersihkan(korpus_mentah)            # sebelum dedup kembar (dipakai snapshot & deteksi omnibus)
cek_snapshot_chunk_id(korpus_full, IFACT_DIR / "snapshot_chunk_id.csv")
korpus, PETA_KEMBAR = dedup_kembar(korpus_full, DOK_KEMBAR)
POS = {c: i for i, c in enumerate(korpus["chunk_id"])}
KORPUS_HASH = _md5("".join(korpus["chunk_id"]) + "".join(korpus["isi"]), 10)
print(f"\nTotal satuan hukum: {len(korpus)} (sebelum dedup kembar: {len(korpus_full)}) | "
      f"Dokumen: {korpus.doc_id.nunique()} | KORPUS_HASH={KORPUS_HASH}")

                      file  baris            doc_id
        korpusPM172018.csv    630    Permen_17_2018
         korpusPM42025.csv      5     Permen_4_2025
         korpusPM52021.csv   1463     Permen_5_2021
         korpusPM82024.csv    246     Permen_8_2024
         korpusPM82026.csv    418     Permen_8_2026
         korpusPM92023.csv    436     Permen_9_2023
       korpusPM_2_2023.csv    323     Permen_2_2023
       korpusPM_2_2025.csv    163     Permen_2_2025
        korpusPP282025.csv   2560        PP_28_2025
        korpusPP462021.csv    611        PP_46_2021
        korpusPP492005.csv     21        PP_49_2005
        korpusPP512005.csv    318        PP_51_2005
        korpusPP522000.csv    417        PP_52_2000
        korpusPP532000.csv    177        PP_53_2000
  korpusPerdirjen32024.csv    781  Perdirjen_3_2024
  korpusPerdirjen72021.csv   1886 Perdirjen_07_2021
     korpusPerppu22022.csv   6607     Perppu_2_2022
korpusPerpres_174_2024.csv    123  Perpres_174_2024
        korp

## §1a — Diagnosis Struktur Dokumen (temuan A1)
Mencari sumber kesalahan rujukan: pasal-hub (kemungkinan lampiran/template yang tertampung di satu
pasal), pasal kosong, dan banyaknya penanda amandemen (ciri dokumen omnibus/pengubah).

In [ ]:
POLA_PENANDA_AMANDEMEN = re.compile(r"\b(?:diubah|dihapus|disisipkan)\b", re.I)


def diagnosis_struktur(df: pd.DataFrame) -> pd.DataFrame:
    baris = []
    for did, g in df.groupby("doc_id"):
        per_pasal = g.groupby(g["pasal"].fillna("-")).size()
        baris.append({
            "doc_id": did, "satuan": len(g), "pasal_unik": int(g["pasal"].nunique()),
            "satuan_tanpa_pasal": int(g["pasal"].isna().sum()),
            "maks_satuan_per_pasal": int(per_pasal.max()), "pasal_terbesar": per_pasal.idxmax(),
            "n_pasal_hub": int((per_pasal >= PASAL_HUB_MIN_SATUAN).sum()),
            "baris_penanda_amandemen": int(g["isi"].str.contains(POLA_PENANDA_AMANDEMEN).sum()),
            "omnibus_dikonfigurasi": did in DOK_OMNIBUS,
        })
    return pd.DataFrame(baris).sort_values("maks_satuan_per_pasal", ascending=False)


diag = diagnosis_struktur(korpus)
print(diag.to_string(index=False))
diag.to_csv(HASIL_DIR / "diagnosis_struktur_dokumen.csv", index=False, encoding="utf-8-sig")

_per_pasal = korpus.groupby(["doc_id", korpus["pasal"].fillna("-")]).size()
PASAL_HUB = {(d, str(p).upper()) for (d, p), n in _per_pasal.items() if n >= PASAL_HUB_MIN_SATUAN}
print(f"\nPasal-hub (≥{PASAL_HUB_MIN_SATUAN} satuan): {sorted(PASAL_HUB)}")
_curiga = diag[(diag["baris_penanda_amandemen"] >= 20) & (~diag["omnibus_dikonfigurasi"])]
if len(_curiga):
    print("[SARAN] Dokumen dengan banyak penanda amandemen tapi belum di DOK_OMNIBUS:",
          _curiga["doc_id"].tolist())

           doc_id  satuan  pasal_unik  satuan_tanpa_pasal  maks_satuan_per_pasal pasal_terbesar  n_pasal_hub  baris_penanda_amandemen  omnibus_dikonfigurasi
Perdirjen_07_2021    1417          75                   0                    684             75            1                        0                  False
        UU_6_2023    7997         390                   0                    532              1            3                     1395                   True
    Permen_8_2026     391           9                   0                    334              9            1                       12                  False
    Permen_2_2025     163           1                   0                    163              1            1                        1                  False
    Permen_2_2023     323          37                   0                    135             37            0                        0                  False
   Permen_17_2018     625          86                   0 

## §1b — Ekstraksi Rujukan v1 / v2 / v3 + Validasi Struktural
- **v1**: logika asli (baseline audit). **v2**: versi 03_a (daftar/rentang, eksternal bernomor, dsb.).
- **v3** = v2 + dua penjaga:
  1. *Konteks amandemen* — pada dokumen omnibus, baris setelah "Beberapa ketentuan dalam UU X … diubah"
     (sampai penanda berikutnya atau BAB berganti) merujuk ke **UU X**, bukan ke dokumen omnibus.
     Rujukan omnibus tanpa konteks ditandai dan (default) tidak masuk KG.
  2. *Rujukan dari pasal-hub/lampiran* ditandai dan (default) tidak masuk KG.

In [ ]:
# ---------------------------- Ekstraksi v1 (baseline pembanding) --------------------------
POLA_INTERNAL_V1 = re.compile(r"Pasal\s+(\d+[A-Z]*)(?:\s+ayat\s*\((\d+)\))?", re.I)
POLA_EKSTERNAL_V1 = re.compile(
    r"(Undang-Undang"
    r"|Peraturan +Pemerintah(?: +Pengganti +Undang-Undang)?"
    r"|Peraturan +Presiden"
    r"|(?:Peraturan +Menteri|Peraturan +Direktur +Jenderal|Keputusan +Menteri)"
    r"(?: +(?!Nomor[ ,])[A-Za-z&,]+)*)"
    r" +Nomor +([0-9]+) +Tahun +([12][0-9][0-9][0-9])", re.I)
JENIS_MAP_V1 = {"undang-undang": "UU", "peraturan pemerintah": "PP",
                "peraturan pemerintah pengganti undang-undang": "Perppu",
                "peraturan presiden": "Perpres", "peraturan menteri": "Permen"}


def ekstrak_rujukan_v1(df: pd.DataFrame):
    internal, eksternal = [], []
    doc_lookup = {}
    for _, r in df.drop_duplicates("doc_id").iterrows():
        doc_lookup[(str(r["jenis"]).lower(), str(r["nomor"]), str(r["tahun"]))] = r["doc_id"]
    pasal_ada = set(zip(df["doc_id"], df["pasal"].astype(str)))
    for _, r in df.iterrows():
        teks = r["isi"]
        for m in POLA_INTERNAL_V1.finditer(teks):
            p_tuju = m.group(1)
            if str(p_tuju) != str(r["pasal"]):
                internal.append({"chunk_id": r["chunk_id"], "doc_id": r["doc_id"],
                                 "pasal_asal": r["pasal"], "pasal_tuju": p_tuju,
                                 "ayat_tuju": m.group(2),
                                 "valid": (r["doc_id"], p_tuju) in pasal_ada,
                                 "ms": m.start(), "me": m.end()})
        for m in POLA_EKSTERNAL_V1.finditer(teks):
            jenis_key = re.sub(r"\s+", " ", m.group(1).lower())
            jenis = JENIS_MAP_V1.get(jenis_key, m.group(1))
            tuju = None
            for (j, n, t), did in doc_lookup.items():
                if n == m.group(2) and t == m.group(3):
                    tuju = did; break
            eksternal.append({"chunk_id": r["chunk_id"], "doc_id": r["doc_id"],
                              "jenis_tuju": jenis, "nomor": m.group(2), "tahun": m.group(3),
                              "doc_tuju": tuju, "dalam_korpus": tuju is not None,
                              "ms": m.start(), "me": m.end()})
    return pd.DataFrame(internal), pd.DataFrame(eksternal)


# ---------------------------- Pola v2/v3 (sama dengan 03_a) -------------------------------
JENIS_ALT = (r"(?P<jenis>Kitab\s+Undang-Undang\s+Hukum\s+\w+"
             r"|Undang-Undang\s+Dasar(?:\s+Negara\s+Republik\s+Indonesia\s+Tahun\s+1945)?"
             r"|Undang-Undang"
             r"|Peraturan\s+Pemerintah\s+Pengganti\s+Undang-Undang"
             r"|Peraturan\s+Pemerintah|Peraturan\s+Presiden|Peraturan\s+Menteri"
             r"|Peraturan\s+Direktur\s+Jenderal|Keputusan\s+Menteri"
             r"|Keputusan\s+Direktur\s+Jenderal)")
SISA = r"(?P<sisa>(?:\s+(?!Nomor\b|No\.?\b|ini\b)[A-Za-z&,.-]+){0,12}?)"
EKS_PENUH = re.compile(
    JENIS_ALT + SISA + r"\s+(?:Nomor|No\.?)\s*(?P<nomor>\d+)\w*(?:/[\w./-]+)?\s+Tahun\s+(?P<tahun>\d{4})",
    re.I)
EKS_EKOR_PENUH = re.compile(
    r"\s*,?\s*(?:(?:pada|dalam|dari|sebagaimana\s+diatur\s+dalam|yang\s+diatur\s+dalam)\s+)?" + EKS_PENUH.pattern,
    re.I)
EKS_EKOR_JENIS = re.compile(
    r"\s*,?\s*(?:(?:pada|dalam|dari|sebagaimana\s+diatur\s+dalam|yang\s+diatur\s+dalam)\s+)?" + JENIS_ALT,
    re.I)
ANCHOR = re.compile(r"\bPasal\s+(\d+[A-Z]{0,2})(?![A-Za-z\d])")
QUAL = re.compile(
    r"(?:\s+ayat\s*\(\d+[A-Za-z]?\)(?:\s*(?:,\s*(?:dan|atau|serta)|,|dan|atau|serta)\s*(?:ayat\s*)?\(\d+[A-Za-z]?\))*"
    r"|\s+huruf\s+[a-z](?![a-z])(?:\s*(?:,\s*(?:dan|atau|serta)|,|dan|atau|serta)\s*[a-z](?![a-z]))*"
    r"|\s+angka\s+\d+(?:\s*(?:,\s*(?:dan|atau|serta)|,|dan|atau|serta)\s*(?:angka\s*)?\d+)*)*")
SAMBUNG = re.compile(
    r"(?P<kon>\s*(?:,\s*(?:dan|serta|atau)|,|dan|serta|atau)\s*|\s+(?:sampai\s+dengan|s\.d\.|sd)\s+)"
    r"(?:Pasal\s+(?P<p1>\d+[A-Z]{0,2})(?![A-Za-z\d])"
    r"|(?P<p2>\d+[A-Z]{0,2})(?![A-Za-z\d])(?!\s*\))"
    r"(?!\s+(?:hari|bulan|tahun|minggu|jam|persen|kali|orang|unit|kHz|MHz|GHz|Hz|dB|%)))")
AYAT_DI = re.compile(r"ayat\s*\((\d+[A-Za-z]?)\)")
POLA_KATA_UBAH = re.compile(r"\b(?:diubah|dihapus|disisipkan|ditambahkan|ditambah)\b", re.I)


def kode_dari_teks(t):
    t = re.sub(r"\s+", " ", t.lower())
    for awal, kode in [("peraturan pemerintah pengganti", "PERPPU"), ("peraturan pemerintah", "PP"),
                       ("peraturan direktur", "PERDIRJEN"), ("peraturan presiden", "PERPRES"),
                       ("peraturan menteri", "PERMEN"), ("keputusan menteri", "KEPMEN"),
                       ("keputusan direktur", "KEPDIRJEN"), ("undang-undang dasar", "UUD"),
                       ("undang-undang", "UU"), ("kitab", "KUH")]:
        if t.startswith(awal):
            return kode
    return "LAIN"


def basis_pasal(p):
    m = re.match(r"\d+", str(p))
    return int(m.group()) if m else 10 ** 6


def deteksi_konteks_amandemen(df: pd.DataFrame, dok_omnibus: set):
    """chunk_id → (kode, nomor, tahun) UU/PP yang sedang diubah, atau None (omnibus tanpa konteks).
    Konteks aktif sejak baris 'ketentuan dalam <Peraturan X> ... diubah/dihapus/disisipkan' sampai penanda
    berikutnya atau BAB berganti. Heuristik ini konservatif: bila ragu → None (tidak masuk KG)."""
    hasil, stat = {}, []
    for did in sorted(dok_omnibus):
        g = df[df["doc_id"] == did]
        if g.empty:
            continue
        aktif, bab_aktif, n_penanda, diubah = None, object(), 0, set()
        for r in g.itertuples():
            bab = _s(r.bab)
            if bab != bab_aktif:
                aktif, bab_aktif = None, bab
            teks = r.isi
            for m in EKS_PENUH.finditer(teks):
                pra = teks[max(0, m.start() - 80):m.start()]
                pasca = teks[m.end():m.end() + 400]
                if re.search(r"\bketentuan\b", pra, re.I) and POLA_KATA_UBAH.search(pasca):
                    aktif = (kode_dari_teks(m.group("jenis")), int(m.group("nomor")), int(m.group("tahun")))
                    n_penanda += 1
                    diubah.add(aktif)
                    break
            hasil[r.chunk_id] = aktif
        n_ktx = sum(1 for c in g["chunk_id"] if hasil.get(c) is not None)
        stat.append({"doc_id": did, "baris": len(g), "baris_berkonteks": n_ktx, "penanda": n_penanda,
                     "peraturan_diubah": "; ".join(f"{k}_{n}_{t}" for k, n, t in sorted(diubah))})
    return hasil, pd.DataFrame(stat)


COL_V2I = ["chunk_id", "doc_id", "pasal_asal", "pasal_tuju", "ayat_tuju", "sifat",
           "doc_tuju", "pola", "valid", "ayat_valid", "omnibus_tanpa_konteks", "dari_hub",
           "konteks_amandemen", "ms", "me"]
COL_V2E = ["chunk_id", "doc_id", "kode_jenis", "nomor", "tahun", "doc_tuju",
           "dalam_korpus", "ambigu_jenis", "rujukan_diri", "ms", "me"]


def ekstrak_rujukan_v2(df: pd.DataFrame, konteks_amandemen: dict | None = None,
                       pasal_hub: set | None = None):
    """v2 (03_a) bila konteks_amandemen & pasal_hub None; v3 bila keduanya diberikan."""
    konteks_amandemen = konteks_amandemen or {}
    pasal_hub = pasal_hub or set()
    pasal_per_doc, ayat_per_pasal, lookup, asli = {}, {}, {}, {}
    for did, g in df.groupby("doc_id"):
        pasal_per_doc[did] = {str(p).upper() for p in g["pasal"].dropna()}
        k = parse_docid(did)
        if k:
            lookup[k] = did
        for p in g["pasal"].dropna():
            asli[(did, str(p).upper())] = str(p)
    for (did, p), g in df.groupby(["doc_id", "pasal"]):
        ayat_per_pasal[(did, str(p).upper())] = {str(a).upper() for a in g["ayat"].dropna()}

    internal, eksternal = [], []
    for r in df.itertuples():
        teks, did = r.isi, r.doc_id
        asal = _s(r.pasal).upper()
        omni = r.chunk_id in konteks_amandemen
        key_amd = konteks_amandemen.get(r.chunk_id)
        dari_hub = (did, asal or "-") in pasal_hub

        anchors = list(ANCHOR.finditer(teks))
        i = 0
        while i < len(anchors):
            a = anchors[i]
            items = [(a.group(1).upper(), False)]
            akhir = QUAL.match(teks, a.end()).end()
            ayat_list = AYAT_DI.findall(teks[a.end():akhir])
            while True:
                m2 = SAMBUNG.match(teks, akhir)
                if not m2:
                    break
                num = (m2.group("p1") or m2.group("p2")).upper()
                kon = m2.group("kon")
                items.append((num, "sampai" in kon or "s.d." in kon or "sd" in kon.split()))
                akhir = QUAL.match(teks, m2.end()).end()
                ayat_list += AYAT_DI.findall(teks[m2.end():akhir])
            i += 1
            while i < len(anchors) and anchors[i].start() < akhir:
                i += 1

            sifat, doc_tuju, akhir_eks = "internal", did, akhir
            mfull = EKS_EKOR_PENUH.match(teks, akhir)
            if mfull:
                k = (kode_dari_teks(mfull.group("jenis")), int(mfull.group("nomor")), int(mfull.group("tahun")))
                sifat, doc_tuju = "eksternal_bernomor", lookup.get(k)
                akhir_eks = mfull.end()
            else:
                mj = EKS_EKOR_JENIS.match(teks, akhir)
                if mj and not re.match(r"\s+ini\b", teks[mj.end("jenis"):], re.I):
                    sifat, doc_tuju = "eksternal_tanpa_nomor", None
                    akhir_eks = mj.end()
                elif omni and key_amd is not None:          # v3: rujukan di dalam blok amandemen
                    sifat, doc_tuju = "internal_amandemen", lookup.get(key_amd)

            target, pola = [], "tunggal"
            for idx, (num, rentang) in enumerate(items):
                if rentang and idx > 0:
                    pola = "rentang"
                    lo, hi = basis_pasal(items[idx - 1][0]), basis_pasal(num)
                    if doc_tuju and doc_tuju in pasal_per_doc and 0 <= hi - lo <= 60:
                        target += sorted(p for p in pasal_per_doc[doc_tuju] if lo <= basis_pasal(p) <= hi)
                    else:
                        target.append(num)
                else:
                    if len(items) > 1 and pola == "tunggal":
                        pola = "daftar"
                    target.append(num)

            for p in dict.fromkeys(target):
                if sifat == "internal" and p == asal:       # rujukan intra-pasal
                    continue
                valid = (p in pasal_per_doc[doc_tuju]) if (doc_tuju and doc_tuju in pasal_per_doc) else False
                ayat_valid = None
                if valid and len(set(target)) == 1 and ayat_list:
                    ada = ayat_per_pasal.get((doc_tuju, p), set())
                    ayat_valid = all(x.upper() in ada for x in ayat_list) if ada else None
                internal.append({
                    "chunk_id": r.chunk_id, "doc_id": did, "pasal_asal": r.pasal,
                    "pasal_tuju": asli.get((doc_tuju, p), p),
                    "ayat_tuju": ",".join(ayat_list) or None, "sifat": sifat,
                    "doc_tuju": doc_tuju, "pola": pola, "valid": bool(valid),
                    "ayat_valid": ayat_valid,
                    "omnibus_tanpa_konteks": bool(omni and key_amd is None and sifat == "internal"),
                    "dari_hub": bool(dari_hub),
                    "konteks_amandemen": "_".join(map(str, key_amd)) if key_amd else None,
                    "ms": a.start(), "me": akhir_eks})

        for m in EKS_PENUH.finditer(teks):
            k = (kode_dari_teks(m.group("jenis")), int(m.group("nomor")), int(m.group("tahun")))
            tuju = lookup.get(k)
            ambigu = tuju is None and any(kk[1:] == k[1:] for kk in lookup)
            eksternal.append({
                "chunk_id": r.chunk_id, "doc_id": did, "kode_jenis": k[0],
                "nomor": k[1], "tahun": k[2], "doc_tuju": tuju,
                "dalam_korpus": tuju is not None, "ambigu_jenis": bool(ambigu),
                "rujukan_diri": tuju == did, "ms": m.start(), "me": m.end()})
    return pd.DataFrame(internal, columns=COL_V2I), pd.DataFrame(eksternal, columns=COL_V2E)


def hasil_str(r) -> str:
    """Representasi target rujukan — format SAMA dengan kolom 'hasil_v2' lembar audit 03_a."""
    doc_tuju = r["doc_tuju"] if pd.notna(r["doc_tuju"]) else "(luar_korpus)"
    ayat = f" ayat ({r['ayat_tuju']})" if pd.notna(r["ayat_tuju"]) else ""
    return f"{r['sifat']} -> {doc_tuju}::P{r['pasal_tuju']}{ayat}"


# Konteks amandemen dideteksi pada korpus SEBELUM dedup kembar (baris penanda bisa saja kembar)
KONTEKS_AMANDEMEN, stat_omnibus = deteksi_konteks_amandemen(korpus_full, DOK_OMNIBUS)
print("=== Deteksi konteks amandemen (dokumen omnibus) ===")
print(stat_omnibus.to_string(index=False) if len(stat_omnibus) else "(tidak ada dokumen omnibus di korpus)")

ref_internal_v1, ref_eksternal_v1 = ekstrak_rujukan_v1(korpus)
ref_internal_v2, ref_eksternal_v2 = ekstrak_rujukan_v2(korpus)
ref_internal_v3, ref_eksternal_v3 = ekstrak_rujukan_v2(korpus, KONTEKS_AMANDEMEN, PASAL_HUB)
for _ri in (ref_internal_v2, ref_internal_v3):
    _ri["hasil"] = [hasil_str(r) for _, r in _ri.iterrows()] if len(_ri) else []


def _n(mask):
    return int(mask.sum())


perbandingan = pd.DataFrame([
    ["Rujukan pasal (total)", len(ref_internal_v1), len(ref_internal_v2), len(ref_internal_v3)],
    ["  murni internal & valid", _n(ref_internal_v1["valid"] == True) if len(ref_internal_v1) else 0,  # noqa: E712
     _n((ref_internal_v2.sifat == "internal") & ref_internal_v2.valid),
     _n((ref_internal_v3.sifat == "internal") & ref_internal_v3.valid)],
    ["  internal_amandemen (v3)", "-", "-", _n(ref_internal_v3.sifat == "internal_amandemen")],
    ["    ↳ target ditemukan di korpus", "-", "-",
     _n((ref_internal_v3.sifat == "internal_amandemen") & ref_internal_v3.valid)],
    ["  omnibus tanpa konteks (v3)", "-", "-", _n(ref_internal_v3.omnibus_tanpa_konteks)],
    ["  berasal dari pasal-hub (v3)", "-", "-", _n(ref_internal_v3.dari_hub)],
    ["  eksternal bernomor", "-", _n(ref_internal_v2.sifat == "eksternal_bernomor"),
     _n(ref_internal_v3.sifat == "eksternal_bernomor")],
    ["  eksternal tanpa nomor", "-", _n(ref_internal_v2.sifat == "eksternal_tanpa_nomor"),
     _n(ref_internal_v3.sifat == "eksternal_tanpa_nomor")],
    ["Rujukan antardokumen (sebutan)", len(ref_eksternal_v1), len(ref_eksternal_v2), len(ref_eksternal_v3)],
    ["  menuju dokumen dalam korpus", _n(ref_eksternal_v1["dalam_korpus"]) if len(ref_eksternal_v1) else 0,
     _n(ref_eksternal_v2.dalam_korpus), _n(ref_eksternal_v3.dalam_korpus)],
], columns=["Ukuran", "v1", "v2 (03_a)", "v3 (03_b)"])
print("\n=== Perbandingan ekstraksi rujukan v1 / v2 / v3 ===")
print(perbandingan.to_string(index=False))

for nama, d in [("ref_internal_v2", ref_internal_v2), ("ref_eksternal_v2", ref_eksternal_v2),
                ("ref_internal_v3", ref_internal_v3), ("ref_eksternal_v3", ref_eksternal_v3)]:
    d.to_csv(IFACT_DIR / f"{nama}.csv", index=False, encoding="utf-8-sig")

=== Deteksi konteks amandemen (dokumen omnibus) ===
       doc_id  baris  baris_berkonteks  penanda                                                                                                                                                                                                                                                                                                                              peraturan_diubah
Perppu_2_2022   6572              4892       30 UU_1_2009; UU_2_1981; UU_3_2004; UU_3_2014; UU_5_1997; UU_5_1999; UU_6_1983; UU_6_2011; UU_6_2014; UU_7_1983; UU_7_1992; UU_7_2016; UU_8_1983; UU_10_1997; UU_11_2019; UU_12_2011; UU_13_2003; UU_13_2016; UU_17_2019; UU_20_2016; UU_21_2008; UU_24_2011; UU_25_1992; UU_33_2014; UU_35_2009; UU_36_1999; UU_38_2009; UU_41_1999; UU_44_2009
    UU_6_2023   7997              4333       17                                                                                                                                         

## §1c — Norma Deontik (v1 = 03_a, v2 = pola diperketat) + Deteksi Intent Kueri
Perubahan v2: `SANKSI` tidak lagi dipicu kata lepas "dikenakan"/"pencabutan" (mis. "dikenakan BHP"),
`IZIN_HAK` hanya "dapat + kata kerja aktif" (03_a: presisi 0,15), `DEKLARATIF` hanya pola definisi
(03_a: "meliputi/terdiri atas" → presisi 0,38).

> **Catatan validitas:** kolom `tipe` juga hasil heuristik parser, jadi kappa di bawah adalah
> kesepakatan antar-heuristik, bukan akurasi. Validasi sesungguhnya butuh sampel berlabel manual.

In [ ]:
POLA_DEONTIK_V1 = {
    "SANKSI": re.compile(r"\b(?:dikenakan|dikenai|dikenakan\s+sanksi|sanksi\s+administratif|pidana|denda|pencabutan)\b", re.I),
    "LARANGAN": re.compile(r"\b(?:dilarang|tidak\s+boleh|tidak\s+diperkenankan|tidak\s+dapat\s+(?:melakukan|mengajukan|diberikan))\b", re.I),
    "KEWAJIBAN": re.compile(r"\b(?:wajib|harus|diharuskan|keharusan)\b", re.I),
    "IZIN_HAK": re.compile(r"\b(?:berhak|berwenang|diberikan\s+hak|diberi\s+izin)\b|"
                           r"\bdapat\b(?!\s+(?:dikenakan|dikenai|diberikan\s+sanksi|dijatuhi))", re.I),
    "DEKLARATIF": re.compile(r"\b(?:merupakan|adalah|yang\s+dimaksud\s+dengan|meliputi|terdiri\s+atas)\b", re.I),
}
POLA_DEONTIK_V2 = {
    "SANKSI": re.compile(r"\b(?:dikenai|dikenakan|dijatuhi)\s+sanksi\b|\bsanksi\s+(?:administratif|pidana)\b|"
                         r"\b(?:dipidana|pidana\s+(?:penjara|kurungan|denda))\b|\bdenda\s+(?:administratif|paling)\b|"
                         r"\bpencabutan\s+izin\b|\bteguran\s+tertulis\b", re.I),
    "LARANGAN": POLA_DEONTIK_V1["LARANGAN"],
    "KEWAJIBAN": re.compile(r"\b(?:wajib|harus|diharuskan|diwajibkan|keharusan)\b", re.I),
    "IZIN_HAK": re.compile(r"\b(?:berhak|berwenang|kewenangan|diberi(?:kan)?\s+hak|diberi\s+izin)\b|"
                           r"\bdapat\s+(?:me|ber|mem|men|meng|meny)\w+", re.I),
    "DEKLARATIF": re.compile(r"\byang\s+dimaksud\s+dengan\b|^[^.;:]{0,120}?\b(?:adalah|merupakan)\b", re.I),
}
_URUT_DEONTIK = ["SANKSI", "LARANGAN", "KEWAJIBAN", "IZIN_HAK", "DEKLARATIF"]
MAPPING_TIPE = {"Definisi": "DEKLARATIF", "Hak": "IZIN_HAK", "Ketentuan": "KETENTUAN_UMUM",
                "Kewajiban": "KEWAJIBAN", "Larangan": "LARANGAN", "Sanksi": "SANKSI"}


def ekstrak_norma_deontik(teks: str, pola: dict) -> tuple:
    if not isinstance(teks, str) or not teks.strip():
        return "KETENTUAN_UMUM", None
    for label in _URUT_DEONTIK:
        m = pola[label].search(teks)
        if m:
            return label, m.group().lower()
    return "KETENTUAN_UMUM", None


for _v, _p in [("v1", POLA_DEONTIK_V1), ("v2", POLA_DEONTIK_V2)]:
    _e = [ekstrak_norma_deontik(t, _p) for t in korpus["isi"]]
    korpus[f"deontik_{_v}"] = [e[0] for e in _e]
    korpus[f"deontik_marker_{_v}"] = [e[1] for e in _e]
korpus["deontik_otomatis"] = korpus[f"deontik_{DEONTIK_AKTIF}"]
korpus["deontik_marker"] = korpus[f"deontik_marker_{DEONTIK_AKTIF}"]

try:
    from sklearn.metrics import classification_report, cohen_kappa_score
    y_true = korpus["tipe"].map(MAPPING_TIPE).fillna("KETENTUAN_UMUM")
    ringkas = []
    for _v in ("v1", "v2"):
        rep = classification_report(y_true, korpus[f"deontik_{_v}"], zero_division=0, output_dict=True)
        ringkas.append({"versi": _v, "kappa": round(cohen_kappa_score(y_true, korpus[f"deontik_{_v}"]), 4),
                        "macro_F1": round(rep["macro avg"]["f1-score"], 4),
                        **{f"P_{c}": round(rep.get(c, {}).get("precision", float("nan")), 3) for c in _URUT_DEONTIK}})
    print("=== Kesepakatan deontik vs kolom `tipe` (heuristik vs heuristik) ===")
    print(pd.DataFrame(ringkas).to_string(index=False))
    print(f"\nClassification report deontik {DEONTIK_AKTIF} (aktif):")
    print(classification_report(y_true, korpus["deontik_otomatis"], zero_division=0))
except Exception as e:
    print("Metrik kesepakatan deontik dilewati:", e)

POLA_INTENT = [
    ("SANKSI", re.compile(r"\b(?:sanksi|denda|pidana|dipidana|hukuman|pencabutan)\b", re.I)),
    ("LARANGAN", re.compile(r"\b(?:dilarang|larangan|tidak\s+boleh)\b", re.I)),
    ("KEWAJIBAN", re.compile(r"\b(?:wajib|kewajiban|diwajibkan|harus)\b", re.I)),
    ("IZIN_HAK", re.compile(r"\b(?:berhak|hak|wewenang|kewenangan)\b", re.I)),
    ("DEKLARATIF", re.compile(r"\b(?:apa\s+yang\s+dimaksud|pengertian|definisi|apa\s+itu)\b", re.I)),
]


def deteksi_intent(query: str) -> set:
    """Modalitas deontik yang ditanyakan kueri (bisa >1, bisa kosong)."""
    return {lab for lab, p in POLA_INTENT if p.search(str(query))}


DEONTIK_DARI_CID = dict(zip(korpus["chunk_id"], korpus["deontik_otomatis"]))

=== Kesepakatan deontik vs kolom `tipe` (heuristik vs heuristik) ===
versi  kappa  macro_F1  P_SANKSI  P_LARANGAN  P_KEWAJIBAN  P_IZIN_HAK  P_DEKLARATIF
   v1 0.6443    0.7135     0.860       0.925        0.933       0.145         0.369
   v2 0.7077    0.7421     0.985       0.909        0.912       0.185         0.515

Classification report deontik v2 (aktif):
                precision    recall  f1-score   support

    DEKLARATIF       0.52      0.77      0.62      1185
      IZIN_HAK       0.19      0.48      0.27       432
KETENTUAN_UMUM       0.94      0.91      0.93     17917
     KEWAJIBAN       0.91      0.81      0.86      1777
      LARANGAN       0.91      0.96      0.94       248
        SANKSI       0.98      0.74      0.84      1995

      accuracy                           0.88     23554
     macro avg       0.74      0.78      0.74     23554
  weighted avg       0.91      0.88      0.89     23554



## §2 — Baseline RAG (Dense): header konteks → embedding → FAISS
Cache embedding kini dikunci dengan hash(model + awalan + seluruh teks) (B6): mengganti model atau
mengubah teks memicu pembangunan ulang otomatis.

In [ ]:
def _awalan(peran: str) -> str:
    if "e5" in EMBED_MODEL_NAME.lower():
        return "query: " if peran == "query" else "passage: "
    return ""


def teks_chunk(r) -> str:
    lokasi = f"Pasal {_s(r['pasal']) or '-'}"
    if pd.notna(r["ayat"]):  lokasi += f" ayat ({r['ayat']})"
    if pd.notna(r["huruf"]): lokasi += f" huruf {r['huruf']}"
    if pd.notna(r["angka"]): lokasi += f" angka {r['angka']}"
    tipe = f" [{r['tipe']}]" if TEKS_EMBED_SERTAKAN_TIPE else ""
    return f"{_s(r['nama_dokumen'])} — {_s(r['nama_bab'])} — {lokasi}{tipe}: {r['isi']}"


def kunci_embedding(model: str, awalan: str, teks: list) -> str:
    h = hashlib.md5((model + "|" + awalan).encode())
    for t in teks:
        h.update(t.encode("utf-8")); h.update(b"\x00")
    return h.hexdigest()[:12]


korpus["teks_embed"] = korpus.apply(teks_chunk, axis=1)
embedder = SentenceTransformer(EMBED_MODEL_NAME)
EMB_KEY = kunci_embedding(EMBED_MODEL_NAME, _awalan("passage"), korpus["teks_embed"].tolist())
EMB_PATH = IFACT_DIR / f"emb_{EMB_KEY}.npy"
IDX_PATH = IFACT_DIR / f"faiss_{EMB_KEY}.index"


def bangun_indeks(force=False):
    if (not force) and EMB_PATH.exists() and IDX_PATH.exists():
        print(f"Cache embedding cocok (kunci {EMB_KEY}) — dimuat.")
        return np.load(EMB_PATH), faiss.read_index(str(IDX_PATH))
    print(f"Membangun embedding baru (kunci {EMB_KEY}) ...")
    teks = [_awalan("passage") + t for t in korpus["teks_embed"]]
    emb = np.asarray(embedder.encode(teks, batch_size=64, show_progress_bar=True,
                                     normalize_embeddings=True), dtype="float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    np.save(EMB_PATH, emb)
    faiss.write_index(index, str(IDX_PATH))
    return emb, index


embeddings, faiss_index = bangun_indeks()
assert len(embeddings) == len(korpus) == faiss_index.ntotal
print("Indeks FAISS:", faiss_index.ntotal, "vektor, dim", embeddings.shape[1])

KOL_HASIL = ["chunk_id", "nama_dokumen", "pasal", "ayat", "tipe", "isi", "skor"]


@lru_cache(maxsize=4096)
def encode_query(query: str) -> np.ndarray:
    v = embedder.encode([_awalan("query") + query], normalize_embeddings=True)
    v = np.asarray(v, dtype="float32")[0]
    v.setflags(write=False)
    return v


def _ambil(idx_list, skor_list) -> pd.DataFrame:
    hasil = korpus.iloc[list(idx_list)].copy()
    hasil["skor"] = list(skor_list)
    return hasil[KOL_HASIL]


def retrieve_dense(query: str, k: int = 10) -> pd.DataFrame:
    skor, idx = faiss_index.search(encode_query(query)[None, :], k)
    return _ambil(idx[0], skor[0])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Cache embedding cocok (kunci 0e8e09a58477) — dimuat.
Indeks FAISS: 23554 vektor, dim 384


## §3 — Hybrid Retrieval: Dense + BM25 (RRF) + filter metadata
Skor BM25 dan vektor kueri di-cache per kueri (evaluasi memanggil kueri yang sama berkali-kali).
Dengan filter metadata, dense dihitung eksak pada subset (03_a bisa kehabisan kandidat).

In [ ]:
STOPWORD = set("""yang dan di ke dari untuk pada dengan dalam adalah atau sebagaimana dimaksud
ayat pasal huruf angka bab ini itu oleh dapat tidak akan telah harus wajib sebagai atas terhadap
dilakukan berdasarkan sesuai antara serta bahwa maka setiap paling""".split())


def tokenisasi(t: str):
    return [w for w in re.findall(r"[a-z0-9]+", str(t).lower()) if w not in STOPWORD and len(w) > 2]


bm25 = BM25Okapi([tokenisasi(t) for t in korpus["teks_embed"]])
print("BM25 siap:", len(korpus), "dokumen")


@lru_cache(maxsize=128)
def _skor_bm25(query: str) -> np.ndarray:
    s = np.asarray(bm25.get_scores(tokenisasi(query)), dtype="float64")
    s.setflags(write=False)
    return s


def retrieve_bm25(query: str, k: int = 10) -> pd.DataFrame:
    skor = _skor_bm25(query)
    top = np.argsort(-skor, kind="stable")[:k]
    return _ambil(top, skor[top])


def saring_mask(df: pd.DataFrame, filter_meta: dict | None) -> np.ndarray:
    if not filter_meta:
        return np.ones(len(df), dtype=bool)
    m = pd.Series(True, index=df.index)
    for kol, nilai in filter_meta.items():
        if kol == "tahun" and isinstance(nilai, tuple):
            m &= df["tahun"].between(*nilai)
        elif kol in ("entitas", "kata_kunci"):
            m &= df[kol].astype(str).str.contains(str(nilai), case=False, na=False)
        elif isinstance(nilai, (list, set, tuple)):
            m &= df[kol].isin(list(nilai))
        else:
            m &= df[kol].astype(str).str.lower() == str(nilai).lower()
    return m.fillna(False).to_numpy(dtype=bool)


def retrieve_hybrid(query: str, k: int = 10, filter_meta: dict | None = None,
                    k_kandidat: int = 50, rrf_k: int = RRF_K) -> pd.DataFrame:
    mask = saring_mask(korpus, filter_meta)
    if not mask.any():
        return korpus.iloc[0:0].assign(skor=[])[KOL_HASIL]
    qv = encode_query(query)
    if filter_meta:
        sub = np.where(mask)[0]
        sims = embeddings[sub] @ qv
        rank_dense = sub[np.argsort(-sims, kind="stable")][:k_kandidat].tolist()
    else:
        _, idx = faiss_index.search(qv[None, :], min(k_kandidat, len(korpus)))
        rank_dense = [int(i) for i in idx[0] if i >= 0]
    skor_bm = _skor_bm25(query)
    rank_bm = [int(i) for i in np.argsort(-skor_bm, kind="stable") if mask[i]][:k_kandidat]
    rrf = {}
    for r, i in enumerate(rank_dense):
        rrf[i] = rrf.get(i, 0.0) + 1 / (rrf_k + r + 1)
    for r, i in enumerate(rank_bm):
        rrf[i] = rrf.get(i, 0.0) + 1 / (rrf_k + r + 1)
    top = sorted(rrf, key=lambda i: (-rrf[i], i))[:k]
    return _ambil(top, [rrf[i] for i in top])

BM25 siap: 23554 dokumen


## §4 — Knowledge Graph (ekstraksi aktif + atribut deontik + penanda hub/entitas generik)
Simpul: dokumen → bab → pasal → satuan, plus entitas. Sisi: `memuat`, `merujuk`, `merujuk_dokumen`,
`menyebut`. Perubahan: rujukan omnibus-tanpa-konteks & dari pasal-hub disaring, entitas generik
ditandai, `merujuk_dokumen` menuju **pasal definisi** dokumen tujuan (bukan P1 buta — di UU 6/2023 P1 adalah hub).

In [ ]:
def id_bab(doc, bab):     return f"{doc}::{_s(bab) or '(tanpa_bab)'}"
def id_pasal(doc, pasal): return f"{doc}::P{_s(pasal) or '-'}"


def mask_kg_internal(ri: pd.DataFrame) -> pd.Series:
    """Baris rujukan pasal yang dipakai sebagai sisi `merujuk` di KG."""
    if EKSTRAKSI_AKTIF == "v1":
        return ri["valid"] == True  # noqa: E712
    m = (ri["valid"] == True) & ri["doc_tuju"].notna()  # noqa: E712
    if EKSTRAKSI_AKTIF == "v3":
        if KG_ABAIKAN_OMNIBUS_TANPA_KONTEKS:
            m &= ~ri["omnibus_tanpa_konteks"].astype(bool)
        if KG_ABAIKAN_RUJUKAN_DARI_HUB:
            m &= ~ri["dari_hub"].astype(bool)
        m &= ~(ri["doc_id"].isin(KG_DOK_TANPA_RUJUKAN_INTERNAL) & (ri["sifat"] == "internal"))
    return m


if EKSTRAKSI_AKTIF == "v1":
    REF_INTERNAL, REF_EKSTERNAL = ref_internal_v1.assign(doc_tuju=lambda t: t["doc_id"]), ref_eksternal_v1
elif EKSTRAKSI_AKTIF == "v2":
    REF_INTERNAL, REF_EKSTERNAL = ref_internal_v2, ref_eksternal_v2
else:
    REF_INTERNAL, REF_EKSTERNAL = ref_internal_v3, ref_eksternal_v3

ref_kg_internal = REF_INTERNAL[mask_kg_internal(REF_INTERNAL)].copy()
_me = REF_EKSTERNAL["dalam_korpus"].astype(bool)
if "rujukan_diri" in REF_EKSTERNAL:
    _me &= ~REF_EKSTERNAL["rujukan_diri"].astype(bool)
ref_kg_eksternal = REF_EKSTERNAL[_me].copy()
print(f"KG memakai ekstraksi {EKSTRAKSI_AKTIF}: {len(ref_kg_internal)} rujukan antarpasal "
      f"(dari {len(REF_INTERNAL)}), {len(ref_kg_eksternal)} rujukan antardokumen (sebelum dedup).")

# Entitas generik
_ent = (korpus["entitas"].dropna().astype(str).str.split(",").explode().str.strip())
_ent = _ent[_ent != ""]
DF_ENTITAS = _ent.groupby(_ent).size().sort_values(ascending=False)
ENT_GENERIK = set(DF_ENTITAS[DF_ENTITAS / len(korpus) > ENT_GENERIK_MAKS_PROPORSI].index)
print(f"Entitas: {len(DF_ENTITAS)} unik | generik (>{ENT_GENERIK_MAKS_PROPORSI:.0%} satuan, diabaikan "
      f"saat ekspansi): {len(ENT_GENERIK)} → {sorted(ENT_GENERIK)[:15]}")


def pilih_pasal_definisi(df: pd.DataFrame) -> dict:
    out = {}
    pola_def = r"yang dimaksud dengan|\badalah\b"
    for did, g in df[df["pasal"].notna()].groupby("doc_id"):
        is_def = (g["tipe"] == "Definisi") | g["isi"].str.contains(pola_def, case=False, regex=True)
        skor = is_def.groupby(g["pasal"]).sum()
        kandidat = [p for p in skor.index if (did, str(p).upper()) not in PASAL_HUB]
        if kandidat:
            best = max(kandidat, key=lambda p: (skor[p], -basis_pasal(p)))
            if skor[best] > 0:
                out[did] = id_pasal(did, best)
    return out


PASAL_DEFINISI = pilih_pasal_definisi(korpus)


def bangun_kg() -> nx.MultiDiGraph:
    KG = nx.MultiDiGraph()
    n_satuan = korpus.groupby(["doc_id", korpus["pasal"].fillna("-")]).size()
    for r in korpus.itertuples():
        d, b, p = r.doc_id, id_bab(r.doc_id, r.bab), id_pasal(r.doc_id, r.pasal)
        if d not in KG:
            KG.add_node(d, tipe="dokumen", label=r.nama_dokumen, jenis=r.jenis,
                        tahun=int(r.tahun) if pd.notna(r.tahun) else None)
        if b not in KG:
            KG.add_node(b, tipe="bab", label=_s(r.nama_bab))
        if p not in KG:
            ps = _s(r.pasal) or "-"
            KG.add_node(p, tipe="pasal", label=f"Pasal {ps}", n_satuan=int(n_satuan[(d, ps)]),
                        hub=(d, ps.upper()) in PASAL_HUB, definisi=PASAL_DEFINISI.get(d) == p)
        KG.add_node(r.chunk_id, tipe="satuan", norma=r.tipe, norma_deontik_auto=r.deontik_otomatis,
                    deontik_marker=r.deontik_marker, pasal=_s(r.pasal))
        KG.add_edge(d, b, key="memuat", rel="memuat")
        KG.add_edge(b, p, key="memuat", rel="memuat")
        KG.add_edge(p, r.chunk_id, key="memuat", rel="memuat")
    n_lewat = 0
    for r in ref_kg_internal.drop_duplicates(["doc_id", "pasal_asal", "doc_tuju", "pasal_tuju"]).itertuples():
        u, v = id_pasal(r.doc_id, r.pasal_asal), id_pasal(r.doc_tuju, r.pasal_tuju)
        if u in KG and v in KG and u != v:          # jangan ciptakan simpul tanpa atribut
            KG.add_edge(u, v, key="merujuk", rel="merujuk")
        else:
            n_lewat += 1
    for r in ref_kg_eksternal.drop_duplicates(["doc_id", "doc_tuju"]).itertuples():
        if pd.notna(r.doc_tuju) and r.doc_id != r.doc_tuju and r.doc_tuju in KG:
            KG.add_edge(r.doc_id, r.doc_tuju, key="merujuk_dokumen", rel="merujuk_dokumen")
    for r in korpus[korpus["entitas"].notna()].itertuples():
        for ent in str(r.entitas).split(","):
            ent = ent.strip()
            if ent:
                eid = f"ENT::{ent}"
                if eid not in KG:
                    KG.add_node(eid, tipe="entitas", label=ent, generik=ent in ENT_GENERIK,
                                df=int(DF_ENTITAS.get(ent, 0)))
                KG.add_edge(r.chunk_id, eid, key="menyebut", rel="menyebut")
    if n_lewat:
        print(f"Sisi merujuk dilewati (simpul ujung tak ada / rujukan diri): {n_lewat}")
    return KG


KG = bangun_kg()
print(f"Simpul: {KG.number_of_nodes():,} | Sisi: {KG.number_of_edges():,}")
print(pd.Series([d["tipe"] for _, d in KG.nodes(data=True)]).value_counts().to_string())
print()
print(pd.Series([d["rel"] for _, _, d in KG.edges(data=True)]).value_counts().to_string())
with open(IFACT_DIR / "knowledge_graph_03b.gpickle", "wb") as f:
    pickle.dump(KG, f)

# ---- Indeks ketetanggaan (dipakai traversal; jauh lebih cepat daripada query KG berulang) ----
SATUAN_PER_PASAL, PASAL_DARI_SATUAN, DOC_DARI_SATUAN = {}, {}, {}
MERUJUK_OUT, MERUJUK_IN, DOK_RUJUKAN = {}, {}, {}
ENT_DARI_SATUAN, SATUAN_DARI_ENT = {}, {}
for u, v, d in KG.edges(data=True):
    rel = d["rel"]
    if rel == "memuat" and KG.nodes[u]["tipe"] == "pasal":
        SATUAN_PER_PASAL.setdefault(u, []).append(v)
        PASAL_DARI_SATUAN[v] = u
        DOC_DARI_SATUAN[v] = u.split("::")[0]
    elif rel == "merujuk":
        MERUJUK_OUT.setdefault(u, set()).add(v)
        MERUJUK_IN.setdefault(v, set()).add(u)
    elif rel == "merujuk_dokumen":
        DOK_RUJUKAN.setdefault(u, []).append(v)
    elif rel == "menyebut":
        ENT_DARI_SATUAN.setdefault(u, []).append(v)
        SATUAN_DARI_ENT.setdefault(v, []).append(u)
ENT_GENERIK_ID = {f"ENT::{e}" for e in ENT_GENERIK}

deg = pd.Series({n: KG.degree(n) for n, d in KG.nodes(data=True) if d["tipe"] == "pasal"})
print("\nPasal paling terhubung (derajat | jumlah satuan | hub?):")
for n, v in deg.sort_values(ascending=False).head(8).items():
    print(f"  {n:<32} derajat {v:<5} satuan {KG.nodes[n]['n_satuan']:<5} hub={KG.nodes[n]['hub']}")


def gambar_subgraf(node_pusat: str, radius: int = 1, out="subgraf.html"):
    try:
        from pyvis.network import Network
    except Exception:
        pip_install("pyvis")
        from pyvis.network import Network
    if node_pusat not in KG:
        print(f"Simpul {node_pusat} tidak ada — dilewati.")
        return
    simpul = {node_pusat}
    for _ in range(radius):
        tetangga = set()
        for s in simpul:
            tetangga |= set(KG.successors(s)) | set(KG.predecessors(s))
        simpul |= tetangga
    sub = KG.subgraph(simpul)
    warna = {"dokumen": "#d62728", "bab": "#ff7f0e", "pasal": "#1f77b4", "satuan": "#2ca02c", "entitas": "#9467bd"}
    net = Network(height="600px", directed=True, notebook=False, cdn_resources="remote")
    for n, d in sub.nodes(data=True):
        net.add_node(n, label=str(d.get("label", n))[:40], color=warna.get(d.get("tipe"), "#999999"),
                     title=d.get("tipe", "?"))
    for u, v, d in sub.edges(data=True):
        net.add_edge(u, v, label=d["rel"])
    net.save_graph(out)
    print("Tersimpan:", out)

KG memakai ekstraksi v3: 1964 rujukan antarpasal (dari 6390), 104 rujukan antardokumen (sebelum dedup).
Entitas: 33 unik | generik (>1% satuan, diabaikan saat ekspansi): 11 → ['Badan', 'Badan Hukum', 'Badan Usaha', 'Lembaga', 'Masyarakat', 'Menteri', 'Pelaku Usaha', 'Pemerintah', 'Pemerintah Daerah', 'Pemerintah Pusat', 'Presiden']
Simpul: 26,386 | Sisi: 39,768
satuan     23554
pasal       2562
bab          214
entitas       33
dokumen       23

memuat             28459
menyebut            9596
merujuk             1694
merujuk_dokumen       19

Pasal paling terhubung (derajat | jumlah satuan | hub?):
  Perdirjen_07_2021::P75           derajat 685   satuan 684   hub=True
  UU_6_2023::P1                    derajat 541   satuan 532   hub=True
  Permen_8_2026::P9                derajat 336   satuan 334   hub=True
  Permen_2_2025::P1                derajat 164   satuan 163   hub=True
  UU_6_2023::P26                   derajat 158   satuan 151   hub=True
  UU_6_2023::P15                   de

In [ ]:
import pandas as pd

# 1. Hitung derajat semua simpul bertipe 'pasal'
deg_pasal = pd.Series({n: KG.degree(n) for n, d in KG.nodes(data=True) if d['tipe'] == 'pasal'})

# 2. Urutkan untuk mengidentifikasi perwakilan derajat
deg_sorted = deg_pasal.sort_values(ascending=False)

# Ambil perwakilan
pasal_tinggi = deg_sorted.index[0]            # Derajat tertinggi
pasal_menengah = deg_sorted.index[len(deg_sorted) // 2]  # Derajat median (menengah)
pasal_rendah = deg_sorted.index[-1]           # Derajat terendah (daun)

print(f"=== Memilih Perwakilan Simpul ===")
print(f"1. Derajat Tinggi   : {pasal_tinggi} (Derajat: {deg_sorted[pasal_tinggi]})")
print(f"2. Derajat Menengah : {pasal_menengah} (Derajat: {deg_sorted[pasal_menengah]})")
print(f"3. Derajat Rendah   : {pasal_rendah} (Derajat: {deg_sorted[pasal_rendah]})")

# 3. Gambar dan simpan subgraf masing-masing
print("\n=== Menghasilkan Visualisasi HTML ===")
gambar_subgraf(pasal_tinggi, radius=1, out="subgraf_derajat_tinggi.html")
gambar_subgraf(pasal_menengah, radius=1, out="subgraf_derajat_menengah.html")
gambar_subgraf(pasal_rendah, radius=1, out="subgraf_derajat_rendah.html")

=== Memilih Perwakilan Simpul ===
1. Derajat Tinggi   : Perdirjen_07_2021::P75 (Derajat: 685)
2. Derajat Menengah : PP_28_2025::P157 (Derajat: 7)
3. Derajat Rendah   : UU_6_2023::P5l (Derajat: 2)

=== Menghasilkan Visualisasi HTML ===
Tersimpan: subgraf_derajat_tinggi.html
Tersimpan: subgraf_derajat_menengah.html
Tersimpan: subgraf_derajat_rendah.html


## §5 — Graph-RAG v3: hybrid → seed → ekspansi berkuota per jenis sisi → fusi RRF
- Tiap kandidat diskor `bobot_relasi × max(0, cosine(kueri, kandidat))` (A4); tiap jenis sisi punya
  kuota sendiri sehingga sisi berbobot rendah tidak selalu tersingkir (A3).
- Hanya `N_POOL_HYBRID` kandidat hybrid ikut fusi, terlepas dari `k` → **top-k stabil terhadap k**
  (diuji di §9).
- Kolom `asal`: `seed` (hybrid 1..K_SEED), `hybrid` (hybrid lainnya), `graf` (hanya dari ekspansi).
  Kolom `rel_graf`: jenis sisi yang menyumbang kandidat.
- Varian deontik: bonus `BETA_DEONTIK/(RRF_K+1)` bila label deontik satuan cocok dengan intent kueri (G1).

In [ ]:
SEMUA_SISI = frozenset({"memuat", "merujuk", "merujuk_dokumen", "menyebut"})


def kandidat_per_relasi(cid: str, aktif_sisi=SEMUA_SISI) -> dict:
    """Kandidat satuan dari satu seed, dikelompokkan per jenis sisi (tanpa kuota)."""
    if cid not in PASAL_DARI_SATUAN:
        return {}
    p, doc = PASAL_DARI_SATUAN[cid], DOC_DARI_SATUAN[cid]
    out = {}
    if "memuat" in aktif_sisi:
        out["memuat"] = SATUAN_PER_PASAL.get(p, [])
    if "merujuk" in aktif_sisi:
        pts = MERUJUK_OUT.get(p, set()) | MERUJUK_IN.get(p, set())
        out["merujuk"] = [c for pt in sorted(pts) for c in SATUAN_PER_PASAL.get(pt, [])]
    if "merujuk_dokumen" in aktif_sisi:
        out["merujuk_dokumen"] = [c for dt in DOK_RUJUKAN.get(doc, [])
                                  for c in SATUAN_PER_PASAL.get(PASAL_DEFINISI.get(dt, ""), [])]
    if "menyebut" in aktif_sisi:
        out["menyebut"] = [c for e in ENT_DARI_SATUAN.get(cid, []) if e not in ENT_GENERIK_ID
                           for c in SATUAN_DARI_ENT.get(e, [])]
    return {rel: [c for c in dict.fromkeys(cs) if c != cid] for rel, cs in out.items()}


def perluas_graf_v3(seed_ids: list, bobot_seed: dict, qv: np.ndarray, aktif_sisi=SEMUA_SISI,
                    kuota: dict | None = None, bobot: dict | None = None, decay: float = DECAY):
    """Return (skor_ekspansi {cid: skor}, rel_asal {cid: relasi penyumbang terbesar})."""
    kuota = kuota or KUOTA_SISI
    bobot = bobot or BOBOT_SISI
    skor, kontrib_rel = {}, {}
    for cid in seed_ids:
        terbaik = {}
        for rel, cands in kandidat_per_relasi(cid, aktif_sisi).items():
            n = kuota.get(rel, 0)
            if not cands or n <= 0:
                continue
            sims = np.clip(embeddings[[POS[c] for c in cands]] @ qv, 0.0, None)
            for j in np.argsort(-sims, kind="stable")[:n]:
                s = bobot[rel] * float(sims[j])
                if s > terbaik.get(cands[j], (0.0, None))[0]:
                    terbaik[cands[j]] = (s, rel)
        for c, (s, rel) in terbaik.items():
            kon = bobot_seed[cid] * decay * s
            skor[c] = skor.get(c, 0.0) + kon            # akumulasi multi-seed
            kontrib_rel.setdefault(c, {}).setdefault(rel, 0.0)
            kontrib_rel[c][rel] += kon
    rel_asal = {c: max(d, key=d.get) for c, d in kontrib_rel.items()}
    return skor, rel_asal


def tambah_bonus_deontik(skor: dict, query: str, beta: float, rrf_k: int = RRF_K) -> dict:
    if beta <= 0:
        return skor
    intent = deteksi_intent(query)
    if not intent:
        return skor
    bonus = beta / (rrf_k + 1)
    return {c: s + (bonus if DEONTIK_DARI_CID.get(c) in intent else 0.0) for c, s in skor.items()}


def _df_hasil(urut: list, skor: dict) -> pd.DataFrame:
    hasil = korpus.iloc[[POS[c] for c in urut]].copy()
    hasil["skor"] = [skor.get(c, 0.0) for c in urut]
    return hasil[KOL_HASIL]


def retrieve_hybrid_deontik(query: str, k: int = 10, beta: float = BETA_DEONTIK,
                            filter_meta: dict | None = None, n_pool: int = 50) -> pd.DataFrame:
    """Hybrid + bonus deontik (pembanding untuk mengisolasi efek deontik tanpa graf).
    Hanya n_pool kandidat teratas yang di-rerank (tetap, tidak bergantung k) → top-k stabil."""
    pool = retrieve_hybrid(query, k=max(k, n_pool), filter_meta=filter_meta)
    inti = pool.head(n_pool)
    skor = tambah_bonus_deontik(dict(zip(inti["chunk_id"], inti["skor"])), query, beta)
    urut = sorted(skor, key=lambda c: (-skor[c], POS[c]))
    urut += [c for c in pool["chunk_id"] if c not in skor]
    skor = {**dict(zip(pool["chunk_id"], pool["skor"])), **skor}
    return _df_hasil(urut[:k], skor)


def retrieve_graphrag(query: str, k: int = 10, k_seed: int = K_SEED, filter_meta: dict | None = None,
                      aktif_sisi=SEMUA_SISI, kuota: dict | None = None, rrf_k: int = RRF_K,
                      alpha_graf: float = ALPHA_GRAF, beta_deontik: float = 0.0,
                      n_pool: int = N_POOL_HYBRID) -> pd.DataFrame:
    pool = retrieve_hybrid(query, k=max(k, n_pool), filter_meta=filter_meta, rrf_k=rrf_k)
    if len(pool) == 0:
        return pool.assign(asal=[], rel_graf=[])
    inti = pool.head(n_pool)
    top_seeds = inti.head(k_seed)
    bobot_seed = dict(zip(top_seeds["chunk_id"], top_seeds["skor"] / top_seeds["skor"].max()))
    skor_eks, rel_asal = perluas_graf_v3(top_seeds["chunk_id"].tolist(), bobot_seed,
                                         encode_query(query), aktif_sisi=aktif_sisi, kuota=kuota)
    if filter_meta:                                     # 03_a: ekspansi bisa keluar dari filter
        mask = saring_mask(korpus, filter_meta)
        skor_eks = {c: s for c, s in skor_eks.items() if mask[POS[c]]}

    skor = dict(zip(inti["chunk_id"], inti["skor"]))
    for rank_g, c in enumerate(sorted(skor_eks, key=lambda c: (-skor_eks[c], POS[c]))):
        skor[c] = skor.get(c, 0.0) + alpha_graf / (rrf_k + rank_g + 1)
    skor = tambah_bonus_deontik(skor, query, beta_deontik, rrf_k)
    urut = sorted(skor, key=lambda c: (-skor[c], POS[c]))
    if len(urut) < k:                                   # padding dari ekor hybrid
        urut += [c for c in pool["chunk_id"] if c not in skor][:k - len(urut)]
    urut = urut[:k]

    hasil = korpus.iloc[[POS[c] for c in urut]].copy()
    hasil["skor"] = [skor.get(c, 0.0) for c in urut]
    seed_set, pool_set = set(top_seeds["chunk_id"]), set(pool["chunk_id"])
    hasil["asal"] = ["seed" if c in seed_set else "hybrid" if c in pool_set else "graf" for c in urut]
    hasil["rel_graf"] = [rel_asal.get(c) if c in skor_eks else None for c in urut]
    return hasil[KOL_HASIL + ["asal", "rel_graf"]]


_q_demo = "Bagaimana ketentuan penggunaan spektrum frekuensi radio tanpa izin?"
hasil_graphrag = retrieve_graphrag(_q_demo, k=5)
print(hasil_graphrag[["chunk_id", "pasal", "tipe", "skor", "asal", "rel_graf"]].to_string(index=False))

                         chunk_id pasal      tipe     skor   asal rel_graf
          PP_46_2021__P64__a1__he    64 Ketentuan 0.044551   seed  merujuk
          PP_46_2021__P63__a1__hb    63 Ketentuan 0.042877   seed  merujuk
              PP_53_2000__P18__a1    18 Ketentuan 0.040110 hybrid  merujuk
Perdirjen_07_2021__P75__hf__n2__0    75  Larangan 0.028283   seed     None
           PP_53_2000__P19__a3__1    19 Ketentuan 0.027056   seed     None


## §6 — Pembangkitan Jawaban LLM + Judge terpisah
Generator dan judge dibuat dengan fungsi yang sama tetapi penyedia/model berbeda (B5). Penyedia `mock`
menghasilkan jawaban deterministik dari konteks — untuk uji alur tanpa API, **bukan untuk hasil tesis**.
Secrets Colab: `GROQ_API_KEY`, `GEMINI_API_KEY`.

In [ ]:
PROMPT_RAG = """Anda adalah asisten hukum untuk regulasi Kementerian Komunikasi dan Digital (Komdigi).
Jawab pertanyaan HANYA berdasarkan konteks regulasi berikut. Setiap klaim WAJIB disertai sitasi dengan
format persis seperti label konteks, yaitu [<Nama Peraturan> — Pasal <n> ayat (<m>)].
Jika konteks tidak memadai, katakan tidak ditemukan.

KONTEKS:
{konteks}

PERTANYAAN: {pertanyaan}

JAWABAN (bahasa Indonesia formal, dengan sitasi):"""


def _ambil_secret(nama):
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(nama)
        except Exception:
            pass
    return os.environ.get(nama)


def _llm_mock(prompt: str) -> str:
    if "FAITHFULNESS" in prompt:
        return '{"faithfulness": 4, "relevancy": 4, "alasan": "mock"}'
    label = re.findall(r"^\[([^\]]+)\]", prompt, re.M)[:3]
    if not label:
        return "Informasi tidak ditemukan dalam konteks."
    return "Berdasarkan konteks, ketentuan terkait diatur dalam " + "; ".join(f"[{x}]" for x in label) + "."


def buat_llm(provider: str, model: str | None = None):
    """Kembalikan (callable(prompt)->str, tag_model) atau (None, None)."""
    try:
        if provider == "mock":
            return _llm_mock, "mock"
        if provider == "groq":
            model = model or GROQ_MODEL
            try:
                from openai import OpenAI
            except ImportError:
                pip_install("openai")
                from openai import OpenAI
            key = _ambil_secret("GROQ_API_KEY")
            if not key:
                raise RuntimeError("GROQ_API_KEY tidak ditemukan")
            client = OpenAI(api_key=key, base_url="https://api.groq.com/openai/v1")

            def _call(prompt):
                resp = client.chat.completions.create(
                    model=model, temperature=0.1, n=1,
                    messages=[{"role": "system", "content": "Anda asisten hukum yang akurat dan tidak mengarang."},
                              {"role": "user", "content": prompt}])
                return resp.choices[0].message.content
            return _call, f"groq:{model}"
        if provider == "gemini":
            model = model or GEMINI_MODEL
            key = _ambil_secret("GEMINI_API_KEY")
            if not key:
                raise RuntimeError("GEMINI_API_KEY tidak ditemukan")
            try:
                from google import genai            # SDK baru (google-generativeai sudah deprecated)
            except ImportError:
                pip_install("google-genai")
                from google import genai
            client = genai.Client(api_key=key)
            return (lambda prompt: client.models.generate_content(model=model, contents=prompt).text,
                    f"gemini:{model}")
        raise ValueError(f"Penyedia tidak dikenal: {provider}")
    except Exception as e:
        print(f"LLM '{provider}' nonaktif: {e}")
        return None, None


LLM, LLM_TAG = buat_llm(LLM_PROVIDER) if USE_LLM else (None, None)
JUDGE, JUDGE_TAG = buat_llm(JUDGE_PROVIDER, JUDGE_MODEL) if USE_LLM else (None, None)
print(f"Generator: {LLM_TAG} | Judge: {JUDGE_TAG}")
if LLM_TAG and LLM_TAG == JUDGE_TAG and LLM_TAG != "mock":
    print("[PERINGATAN] Generator = judge → bias self-evaluation. Laporkan sebagai keterbatasan.")
if "mock" in (LLM_TAG, JUDGE_TAG):
    print("[PERINGATAN] Penyedia MOCK aktif — hasil generasi hanya untuk uji alur, BUKAN hasil penelitian.")


def rakit_konteks(df_hasil: pd.DataFrame, max_char: int = KONTEKS_MAX_CHAR) -> str:
    """Satuan utuh ditambahkan sampai batas max_char (tidak memotong di tengah)."""
    baris, total = [], 0
    for _, r in df_hasil.iterrows():
        lok = f"Pasal {_s(r['pasal']) or '-'}" + (f" ayat ({r['ayat']})" if pd.notna(r.get("ayat")) else "")
        b = f"[{r.get('nama_dokumen', r['chunk_id'])} — {lok}] {r['isi']}"
        if baris and total + len(b) > max_char:
            break
        baris.append(b)
        total += len(b) + 2
    return "\n\n".join(baris)


def jawab(pertanyaan: str, retriever, k: int = 8, llm=None) -> dict:
    llm = LLM if llm is None else llm
    t0 = time.time()
    hasil = retriever(pertanyaan, k)
    t_ret = time.time() - t0
    konteks = rakit_konteks(hasil)
    ok, t_llm = False, 0.0
    if llm is not None:
        t0 = time.time()
        try:
            out = llm(PROMPT_RAG.format(konteks=konteks, pertanyaan=pertanyaan))
            ok = isinstance(out, str) and len(out.strip()) > 0
            if not ok:
                out = "(LLM mengembalikan jawaban kosong)"
        except Exception as e:
            out = f"(LLM gagal: {e})"
        t_llm = time.time() - t0
    else:
        out = "(LLM nonaktif)"
    return {"jawaban": out, "llm_ok": ok, "konteks": hasil, "konteks_str": konteks,
            "waktu_retrieval": t_ret, "waktu_llm": t_llm}


if LLM is not None:
    _demo = jawab("Bagaimana ketentuan penggunaan spektrum frekuensi radio tanpa izin dan sanksinya?",
                  retrieve_graphrag, k=5)
    print("\n=== Contoh jawaban Graph-RAG ===")
    print(_demo["jawaban"][:1200])
    print(f"[ok={_demo['llm_ok']} | retrieval {_demo['waktu_retrieval']:.2f}s | LLM {_demo['waktu_llm']:.2f}s]")

Generator: groq:openai/gpt-oss-120b | Judge: gemini:gemini-3.5-flash-lite

=== Contoh jawaban Graph-RAG ===
Berdasarkan **Perdirjen Nomor 07 Tahun 2021 — Pasal 75**, penggunaan spektrum frekuensi radio **tanpa Izin Stasiun Radio (ISR)** merupakan pelanggaran yang dapat terdeteksi melalui hasil pengawasan. Hal yang sama juga berlaku untuk penggunaan yang menimbulkan gangguan berbahaya (harmful interference) atau yang tidak sesuai peruntukan [Perdirjen Nomor 07 Tahun 2021 — Pasal 75].

Namun, dalam rangkaian regulasi yang disediakan tidak terdapat ketentuan yang secara eksplisit mengatur **sanksi** bagi pelanggaran penggunaan spektrum tanpa izin. Klausul mengenai **permohonan penghentian izin** ([PP Nomor 46 Tahun 2021 — Pasal 63 ayat (1)]) dan **pengalihan izin tanpa persetujuan Menteri** ([PP Nomor 46 Tahun 2021 — Pasal 64 ayat (1)]) mengatur prosedur administratif terkait izin, tetapi tidak menyebutkan jenis sanksi (misalnya denda, pencabutan izin, atau tindakan pidana) yang dapat dik

## §7 — Evaluasi Retrieval (seluruh gold, 6 sistem, 2 granularitas, CI bootstrap, Wilcoxon+Holm, ablasi)
- Gold divalidasi: ID yang tidak ada dilaporkan; ID dokumen kembar diremap (§1).
- Retrieval dijalankan **sekali** per (sistem, kueri) dengan `K_POOL` kandidat, lalu dipakai untuk
  kedua granularitas. Tingkat pasal: dedup pasal dari pool, baru dipotong k (B9).
- Effect size Wilcoxon: r = (W+ − W−)/(W+ + W−); positif = sistem pertama lebih baik (B1).

In [ ]:
from scipy.stats import wilcoxon, rankdata


def muat_gold(path: Path = GOLD_PATH):
    if not path.exists():
        pd.DataFrame({"pertanyaan": ["Apa sanksi bagi penyelenggara telekomunikasi tanpa izin?"],
                      "relevan": ["ISI_CHUNK_ID_1;ISI_CHUNK_ID_2"]}).to_csv(path, index=False)
        print(f"Template gold standard dibuat → {path}. Isi lalu jalankan ulang.")
        return None
    gold = pd.read_csv(path)
    gold = gold[~gold["relevan"].astype(str).str.contains("ISI_CHUNK_ID", na=True)]
    print(f"Gold standard dimuat: {len(gold)} pertanyaan")
    return gold


def validasi_gold(gold: pd.DataFrame, peta_kembar: dict) -> pd.DataFrame:
    ada = set(korpus["chunk_id"])
    laporan, baris = [], []
    for qid, g in gold.iterrows():
        baru = []
        for x in [x.strip() for x in str(g["relevan"]).split(";") if x.strip()]:
            if x in ada:
                baru.append(x); st = "ok"
            elif x in peta_kembar:
                baru.append(peta_kembar[x]); st = f"remap→{peta_kembar[x]}"
            else:
                st = "TIDAK_ADA"
            laporan.append({"kueri_id": qid, "chunk_id": x, "status": st})
        baru = list(dict.fromkeys(baru))
        if baru:
            baris.append({**g.to_dict(), "relevan": ";".join(baru), "kueri_id": qid})
    lap = pd.DataFrame(laporan)
    lap.to_csv(HASIL_DIR / "validasi_gold.csv", index=False, encoding="utf-8-sig")
    n_hilang = int((lap["status"] == "TIDAK_ADA").sum())
    n_remap = int(lap["status"].str.startswith("remap").sum())
    print(f"Validasi gold: {len(lap)} ID | {n_remap} diremap (kembar) | {n_hilang} TIDAK ADA di korpus")
    if n_hilang:
        print(lap[lap["status"] == "TIDAK_ADA"].head(20).to_string(index=False))
    out = pd.DataFrame(baris).set_index("kueri_id")
    if len(out) < len(gold):
        print(f"[PERINGATAN] {len(gold) - len(out)} kueri dibuang karena tidak punya ID relevan yang valid.")
    return out


def cid_ke_pasal(cid: str) -> str:
    parts = str(cid).split("__")
    return "__".join(parts[:2]) if len(parts) >= 2 else str(cid)


def hitung_ndcg(pred: list, relevan: set, k: int) -> float:
    dcg = sum(1.0 / math.log2(i + 2) for i, x in enumerate(pred[:k]) if x in relevan)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevan), k)))
    return dcg / idcg if idcg > 0 else 0.0


def jalankan_retrieval(gold: pd.DataFrame, sistem: dict, k_pool: int = K_POOL) -> dict:
    runs = {}
    for nama, retriever in sistem.items():
        for qid, g in gold.iterrows():
            t0 = time.time()
            pred = retriever(g["pertanyaan"], k_pool)["chunk_id"].tolist()
            runs[(nama, qid)] = (pred, time.time() - t0)
    return runs


def hitung_metrik(runs: dict, gold: pd.DataFrame, granularitas: str = "satuan", k_list=K_LIST) -> pd.DataFrame:
    baris = []
    for (nama, qid), (pred, waktu) in runs.items():
        raw_rel = [x for x in str(gold.loc[qid, "relevan"]).split(";") if x]
        if granularitas == "pasal":
            relevan = {cid_ke_pasal(x) for x in raw_rel}
            hasil = list(dict.fromkeys(cid_ke_pasal(x) for x in pred))
        else:
            relevan, hasil = set(raw_rel), pred
        rr = next((1.0 / i for i, x in enumerate(hasil, 1) if x in relevan), 0.0)
        rec = {"kueri_id": qid, "sistem": nama, "MRR": rr, "waktu_s": waktu}
        for k in k_list:
            top = hasil[:k]
            tp = len(set(top) & relevan)
            rec[f"Recall@{k}"] = tp / len(relevan) if relevan else 0.0
            rec[f"Precision@{k}"] = tp / k
            rec[f"Hit@{k}"] = float(tp > 0)
            rec[f"nDCG@{k}"] = hitung_ndcg(top, relevan, k)
        baris.append(rec)
    return pd.DataFrame(baris)


def ringkas_ci(detail: pd.DataFrame, metrik: list, B: int = BOOTSTRAP_B) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_SEED)
    out = {}
    for nama, g in detail.groupby("sistem", sort=True):
        baris = {}
        n = len(g)
        idx = rng.integers(0, n, size=(B, n))
        for m in metrik:
            x = g[m].to_numpy(float)
            bm = x[idx].mean(axis=1)
            baris[m] = f"{x.mean():.4f} [{np.percentile(bm, 2.5):.3f}–{np.percentile(bm, 97.5):.3f}]"
        out[nama] = baris
    return pd.DataFrame(out).T


def holm(pvals) -> np.ndarray:
    p = np.asarray(pvals, float)
    m, urut = len(p), np.argsort(p)
    adj, jalan = np.empty(m), 0.0
    for i, j in enumerate(urut):
        jalan = max(jalan, (m - i) * p[j])
        adj[j] = min(1.0, jalan)
    return adj


def uji_wilcoxon(detail: pd.DataFrame, pasangan: list, metrik: str = "nDCG@10") -> pd.DataFrame:
    piv = detail.pivot(index="kueri_id", columns="sistem", values=metrik)
    rows = []
    for a, b in pasangan:
        if a not in piv or b not in piv:
            continue
        d = (piv[a] - piv[b]).to_numpy(float)
        dn = d[np.abs(d) > 1e-12]
        if len(dn) == 0:
            wp = wm = 0.0; p = 1.0; r = 0.0
        else:
            rk = rankdata(np.abs(dn))
            wp, wm = float(rk[dn > 0].sum()), float(rk[dn < 0].sum())
            p = float(wilcoxon(dn).pvalue)
            r = (wp - wm) / (wp + wm)
        rows.append({"Perbandingan": f"{a} vs {b}", "Metrik": metrik, "Delta_Mean": round(float(d.mean()), 4),
                     "n_beda": len(dn), "W+": wp, "W-": wm, "p_value": round(p, 4), "r_rb": round(r, 4)})
    res = pd.DataFrame(rows)
    if len(res):
        res["p_holm"] = holm(res["p_value"]).round(4)
        res["signifikan(α=.05,Holm)"] = res["p_holm"] < 0.05
    return res


def jangkauan_per_relasi(seeds: list) -> dict:
    """Semua satuan yang terjangkau 1 langkah traversal dari seed, per jenis sisi (tanpa kuota)."""
    out = {rel: set() for rel in SEMUA_SISI}
    for s in seeds:
        for rel, cands in kandidat_per_relasi(s).items():
            out[rel].update(cands)
    return out


def analisis_kontribusi_traversal(gold: pd.DataFrame, k: int = 10):
    kom, reach = [], []
    for qid, g in gold.iterrows():
        rel = set(str(g["relevan"]).split(";"))
        h = retrieve_graphrag(g["pertanyaan"], k=k)
        for r in h.itertuples():
            kom.append({"kueri_id": qid, "asal": r.asal, "rel_graf": _s(r.rel_graf) or "-", "relevan": r.chunk_id in rel})
        pool = retrieve_hybrid(g["pertanyaan"], k=N_POOL_HYBRID)["chunk_id"].tolist()
        jk = jangkauan_per_relasi(pool[:K_SEED])
        for t in rel:
            reach.append({"kueri_id": qid, "gold": t, "di_seed": t in pool[:K_SEED], "di_pool_hybrid": t in pool,
                          **{f"via_{r}": t in s for r, s in jk.items()}})
    kom, reach = pd.DataFrame(kom), pd.DataFrame(reach)
    via = [c for c in reach.columns if c.startswith("via_")]
    reach["via_graf_mana_saja"] = reach[via].any(axis=1)
    reach["hanya_via_graf"] = reach["via_graf_mana_saja"] & ~reach["di_pool_hybrid"]
    print(f"\n=== KONTRIBUSI TRAVERSAL (Graph-RAG top-{k}) ===")
    t = kom.groupby("asal")["relevan"].agg(jumlah="size", relevan="sum", presisi="mean")
    t["porsi"] = (t["jumlah"] / t["jumlah"].sum()).round(3)
    print(t.round(3).to_string())
    print("\nItem asal 'graf' per jenis sisi:")
    print(kom[kom["asal"] == "graf"].groupby("rel_graf")["relevan"]
          .agg(jumlah="size", relevan="sum", presisi="mean").round(3).to_string())
    print(f"\n=== JANGKAUAN GOLD dari {K_SEED} seed (1 langkah, tanpa kuota) — n_gold={len(reach)} ===")
    print((reach[["di_seed", "di_pool_hybrid"] + via + ["via_graf_mana_saja", "hanya_via_graf"]].mean() * 100)
          .round(2).rename("%").to_string())
    kom.to_csv(HASIL_DIR / "kontribusi_traversal_komposisi.csv", index=False, encoding="utf-8-sig")
    reach.to_csv(HASIL_DIR / "kontribusi_traversal_jangkauan.csv", index=False, encoding="utf-8-sig")
    return kom, reach


SISTEM_UTAMA = {
    "1. BM25 (Sparse)": retrieve_bm25,
    "2. Dense (FAISS)": retrieve_dense,
    "3. Hybrid (RRF)": retrieve_hybrid,
    "4. Graph-RAG": retrieve_graphrag,
    "5. Hybrid + Deontik": lambda q, k: retrieve_hybrid_deontik(q, k, beta=BETA_DEONTIK),
    "6. Graph-RAG + Deontik": lambda q, k: retrieve_graphrag(q, k, beta_deontik=BETA_DEONTIK),
}
PASANGAN_UJI = [("4. Graph-RAG", "1. BM25 (Sparse)"), ("4. Graph-RAG", "2. Dense (FAISS)"),
                ("4. Graph-RAG", "3. Hybrid (RRF)"), ("6. Graph-RAG + Deontik", "4. Graph-RAG"),
                ("5. Hybrid + Deontik", "3. Hybrid (RRF)")]
KOLOM_TAMPIL = ["MRR", "nDCG@10", "Recall@5", "Hit@5", "Recall@10", "Precision@10", "Hit@10"]

gold_raw = muat_gold()
gold = validasi_gold(gold_raw, PETA_KEMBAR) if gold_raw is not None else None

if RUN_EVAL_RETRIEVAL and gold is not None and len(gold):
    print(f"\nIntent deontik terdeteksi pada {sum(bool(deteksi_intent(q)) for q in gold['pertanyaan'])}"
          f"/{len(gold)} kueri gold.")
    runs = jalankan_retrieval(gold, SISTEM_UTAMA, K_POOL)
    detail_satuan = hitung_metrik(runs, gold, "satuan")
    detail_pasal = hitung_metrik(runs, gold, "pasal")
    for judul, det in [("A. GRANULARITAS SATUAN HUKUM (ketat)", detail_satuan),
                       ("B. GRANULARITAS PASAL", detail_pasal)]:
        print(f"\n=== {judul} — rata-rata [CI95 bootstrap], n={det.kueri_id.nunique()} kueri ===")
        print(ringkas_ci(det, KOLOM_TAMPIL).to_string())
    detail_satuan.to_csv(HASIL_DIR / f"detail_satuan_{TIMESTAMP}.csv", index=False, encoding="utf-8-sig")
    detail_pasal.to_csv(HASIL_DIR / f"detail_pasal_{TIMESTAMP}.csv", index=False, encoding="utf-8-sig")

    print("\n=== C. UJI WILCOXON SIGNED-RANK + HOLM ===")
    uji = pd.concat([uji_wilcoxon(det, PASANGAN_UJI, m).assign(Granularitas=gr)
                     for gr, det in [("satuan", detail_satuan), ("pasal", detail_pasal)]
                     for m in ("nDCG@10", "MRR")], ignore_index=True)
    print(uji.to_string(index=False))
    uji.to_csv(HASIL_DIR / f"uji_wilcoxon_{TIMESTAMP}.csv", index=False, encoding="utf-8-sig")

    print("\n=== D. STUDI ABLASI KELUARGA SISI (RQ3) ===")
    SISTEM_ABLASI = {"Graph-RAG (4 sisi)": lambda q, k: retrieve_graphrag(q, k)}
    for s in sorted(SEMUA_SISI):
        SISTEM_ABLASI[f"w/o {s}"] = (lambda s_: lambda q, k: retrieve_graphrag(q, k, aktif_sisi=SEMUA_SISI - {s_}))(s)
    SISTEM_ABLASI["tanpa graf (= Hybrid)"] = retrieve_hybrid
    det_abl = hitung_metrik(jalankan_retrieval(gold, SISTEM_ABLASI, K_POOL), gold, "satuan")
    tbl_abl = det_abl.drop(columns="kueri_id").groupby("sistem").mean(numeric_only=True)
    print(tbl_abl[["MRR", "nDCG@10", "Recall@5", "Recall@10", "Precision@10"]].round(4).to_string())
    print(uji_wilcoxon(det_abl, [("Graph-RAG (4 sisi)", f"w/o {s}") for s in sorted(SEMUA_SISI)]).to_string(index=False))
    det_abl.to_csv(HASIL_DIR / f"detail_ablasi_{TIMESTAMP}.csv", index=False, encoding="utf-8-sig")

    kom_trav, reach_trav = analisis_kontribusi_traversal(gold)

    manifest = {
        "versi_kode": VERSI_KODE, "timestamp": TIMESTAMP, "python": sys.version.split()[0],
        "paket": {"pandas": pd.__version__, "numpy": np.__version__, "networkx": nx.__version__,
                  "faiss": getattr(faiss, "__version__", "?")},
        "korpus_hash": KORPUS_HASH, "n_satuan": len(korpus), "emb_key": EMB_KEY,
        "embed_model": EMBED_MODEL_NAME, "ekstraksi": EKSTRAKSI_AKTIF, "deontik": DEONTIK_AKTIF,
        "n_gold": int(len(gold)), "kuota_sisi": KUOTA_SISI, "bobot_sisi": BOBOT_SISI,
        "k_seed": K_SEED, "n_pool_hybrid": N_POOL_HYBRID, "alpha_graf": ALPHA_GRAF, "beta_deontik": BETA_DEONTIK,
        "dok_kembar": DOK_KEMBAR, "dok_omnibus": sorted(DOK_OMNIBUS),
    }
    (HASIL_DIR / f"manifest_{TIMESTAMP}.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    print(f"\nManifest & tabel tersimpan di {HASIL_DIR}")
else:
    print("Evaluasi retrieval dilewati (gold tidak tersedia atau RUN_EVAL_RETRIEVAL=False).")

Gold standard dimuat: 46 pertanyaan
Validasi gold: 377 ID | 0 diremap (kembar) | 2 TIDAK ADA di korpus
 kueri_id            chunk_id    status
        0   PP_52_2000__P2__0 TIDAK_ADA
        0 PP_28_2025__P172__0 TIDAK_ADA

Intent deontik terdeteksi pada 24/46 kueri gold.

=== A. GRANULARITAS SATUAN HUKUM (ketat) — rata-rata [CI95 bootstrap], n=46 kueri ===
                                         MRR               nDCG@10              Recall@5                 Hit@5             Recall@10          Precision@10                Hit@10
1. BM25 (Sparse)        0.5759 [0.464–0.685]  0.3472 [0.278–0.417]  0.2201 [0.167–0.277]  0.7826 [0.652–0.891]  0.3293 [0.263–0.399]  0.2478 [0.200–0.298]  0.8261 [0.717–0.913]
2. Dense (FAISS)        0.4514 [0.333–0.576]  0.2362 [0.172–0.303]  0.1538 [0.109–0.201]  0.6304 [0.478–0.783]  0.2182 [0.157–0.285]  0.1652 [0.120–0.213]  0.6957 [0.565–0.826]
3. Hybrid (RRF)         0.6265 [0.508–0.739]  0.3304 [0.272–0.390]  0.2078 [0.160–0.258]  0.8478 [0.739–0.935

### §7b — Evaluasi Generasi: Legal Citation F1 (ketat & longgar) + LLM-as-a-Judge ber-checkpoint
- **Ketat**: sitasi dihitung benar bila (dokumen, pasal) cocok dengan gold. Dokumen kembar dianggap sama.
- **Longgar**: hanya nomor pasal (= metrik 03_a), dilaporkan untuk pembanding.
- Checkpoint dikunci hash(sistem, pertanyaan, konfigurasi generator+korpus). Jawaban gagal **tidak**
  disimpan; penilaian judge yang gagal/belum ada diulang tanpa membangkitkan ulang jawaban.

In [ ]:
POLA_SITASI_DOK = re.compile(
    r"\b(?P<jenis>Peraturan\s+Pemerintah\s+Pengganti\s+Undang-Undang|Peraturan\s+Pemerintah|Peraturan\s+Presiden"
    r"|Peraturan\s+Menteri(?:\s+[A-Za-z]+){0,6}?|Peraturan\s+Direktur\s+Jenderal(?:\s+[A-Za-z]+){0,6}?"
    r"|Undang-Undang|Perppu|Perpres|Permen\w*|Perdirjen\w*|PP|UU)"
    r"\s+(?:Nomor|No\.?)\s*0*(?P<nomor>\d+)\w*(?:/[\w./-]+)?\s+Tahun\s+(?P<tahun>\d{4})", re.I)


def _kode_sitasi(jenis: str) -> str:
    j = re.sub(r"\s+", " ", jenis.lower())
    for awal, kode in [("perppu", "PERPPU"), ("peraturan pemerintah pengganti", "PERPPU"),
                       ("perpres", "PERPRES"), ("peraturan presiden", "PERPRES"),
                       ("perdirjen", "PERDIRJEN"), ("peraturan direktur", "PERDIRJEN"),
                       ("permen", "PERMEN"), ("peraturan menteri", "PERMEN"),
                       ("peraturan pemerintah", "PP"), ("undang-undang", "UU")]:
        if j.startswith(awal):
            return kode
    return {"pp": "PP", "uu": "UU"}.get(j, "LAIN")


_KANON = {parse_docid(a): parse_docid(b) for a, b in DOK_KEMBAR.items() if parse_docid(a) and parse_docid(b)}


def kanon(key):
    return _KANON.get(key, key)


def ekstrak_sitasi(teks: str) -> set:
    """{(kunci_dokumen | None, pasal)}. Pola 'Pasal N … <Dokumen>' atau '<Dokumen> … Pasal N'."""
    teks = str(teks)
    docs = [(m.start(), m.end(), kanon((_kode_sitasi(m.group("jenis")), int(m.group("nomor")), int(m.group("tahun")))))
            for m in POLA_SITASI_DOK.finditer(teks)]
    out = set()
    for pm in ANCHOR.finditer(teks):
        key = None
        sesudah = [d for d in docs if 0 <= d[0] - pm.end() <= 60 and not ANCHOR.search(teks[pm.end():d[0]])]
        if sesudah:
            key = sesudah[0][2]
        else:
            sebelum = [d for d in docs if 0 <= pm.start() - d[1] <= 120]
            if sebelum:
                key = sebelum[-1][2]
        out.add((key, pm.group(1).upper()))
    return out


def sitasi_gold(gold_ids: str) -> set:
    out = set()
    for cid in str(gold_ids).split(";"):
        parts = cid.strip().split("__")
        if len(parts) >= 2 and parts[1].startswith("P") and parts[1] != "P-":
            out.add((kanon(parse_docid(parts[0])), parts[1][1:].upper()))
    return out


def _prf(pred: set, gold_set: set) -> tuple:
    if not pred or not gold_set:
        return 0.0, 0.0, 0.0
    tp = len(pred & gold_set)
    p, r = tp / len(pred), tp / len(gold_set)
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def hitung_legal_citation_f1(teks_jawaban: str, gold_ids: str) -> dict:
    pred, g = ekstrak_sitasi(teks_jawaban), sitasi_gold(gold_ids)
    pk, rk, fk = _prf({x for x in pred if x[0] is not None}, g)
    # sitasi tanpa dokumen tetap dihitung sebagai prediksi (tidak dapat diverifikasi) → menurunkan presisi ketat
    if pred and any(x[0] is None for x in pred):
        tp = len(pred & g)
        pk = tp / len(pred)
        fk = 2 * pk * rk / (pk + rk) if pk + rk else 0.0
    pl, rl, fl = _prf({x[1] for x in pred}, {x[1] for x in g})
    return {"cit_p_ketat": pk, "cit_r_ketat": rk, "cit_f1_ketat": fk,
            "cit_p_longgar": pl, "cit_r_longgar": rl, "cit_f1_longgar": fl,
            "n_sitasi": len(pred), "n_sitasi_tanpa_dok": sum(1 for x in pred if x[0] is None)}


PROMPT_JUDGE = """Anda adalah auditor evaluasi sistem AI hukum.
Berikan penilaian objektif dalam skala bulat 1 sampai 5 untuk:
1. FAITHFULNESS: Apakah seluruh klaim dan sitasi pasal dalam JAWABAN didukung penuh oleh KONTEKS tanpa mengarang? (1 = mengarang bebas, 5 = 100% setia pada konteks).
2. RELEVANCY: Apakah JAWABAN menjawab langsung inti PERTANYAAN secara relevan dan jelas? (1 = tidak nyambung, 5 = sangat relevan dan lengkap).

PERTANYAAN: {pertanyaan}
KONTEKS: {konteks}
JAWABAN: {jawaban}

Balas HANYA dalam format JSON valid berikut tanpa teks tambahan:
{{"faithfulness": <int 1-5>, "relevancy": <int 1-5>, "alasan": "<singkat>"}}"""

KOLOM_CKPT = ["kunci", "sistem", "kueri_id", "pertanyaan", "jawaban", "konteks_str", "gen_tag",
              "cit_p_ketat", "cit_r_ketat", "cit_f1_ketat", "cit_p_longgar", "cit_r_longgar", "cit_f1_longgar",
              "n_sitasi", "n_sitasi_tanpa_dok", "faithfulness", "relevancy", "alasan", "judge_tag", "status_judge"]


def _gen_tag(llm_tag) -> str:
    return "|".join([VERSI_KODE, KORPUS_HASH, EMB_KEY, str(llm_tag), str(KONTEKS_MAX_CHAR)])


def _nilai_judge(judge, pertanyaan, konteks_str, jawaban):
    try:
        raw = judge(PROMPT_JUDGE.format(pertanyaan=pertanyaan, konteks=konteks_str, jawaban=jawaban))
        m = re.search(r"\{.*\}", str(raw), re.S)
        if not m:
            return np.nan, np.nan, "", "gagal_parse"
        js = json.loads(m.group(0))
        f, r = float(js["faithfulness"]), float(js["relevancy"])
        if not (1 <= f <= 5 and 1 <= r <= 5):
            return np.nan, np.nan, "", "di_luar_skala"
        return f, r, str(js.get("alasan", "")), "ok"
    except Exception as e:
        return np.nan, np.nan, str(e)[:200], "gagal_api"


def evaluasi_generasi(gold: pd.DataFrame, sistem: dict, k: int = 5, llm=None, llm_tag=None,
                      judge="auto", judge_tag="auto", ckpt_path: Path | None = None,
                      jeda: float = JEDA_API_DETIK, maks_gagal_beruntun: int = 3,
                      verbose: bool = True) -> pd.DataFrame:
    llm = LLM if llm is None else llm
    llm_tag = llm_tag or LLM_TAG
    if isinstance(judge, str) and judge == "auto":        # default: judge global dari §6
        judge, judge_tag = JUDGE, JUDGE_TAG
    ckpt_path = ckpt_path or (IFACT_DIR / "checkpoint_generasi_03b.csv")
    gtag = _gen_tag(llm_tag)
    ck = pd.read_csv(ckpt_path) if ckpt_path.exists() else pd.DataFrame(columns=KOLOM_CKPT)
    for c in KOLOM_CKPT:
        if c not in ck:
            ck[c] = np.nan
    ck = ck.astype({"status_judge": "object", "judge_tag": "object", "alasan": "object"})
    simpan = lambda: ck.to_csv(ckpt_path, index=False, encoding="utf-8-sig")  # noqa: E731
    pakai_jeda = "mock" not in str(llm_tag)

    gagal = 0
    for nama, retriever in sistem.items():
        for qid, g in gold.iterrows():
            kunci = _md5(f"{nama}|{g['pertanyaan']}|{gtag}", 16)
            if (ck["kunci"] == kunci).any():
                continue
            res = jawab(g["pertanyaan"], retriever, k=k, llm=llm)
            if not res["llm_ok"]:
                gagal += 1
                if verbose:
                    print(f"[{nama} | kueri {qid}] generator gagal → tidak disimpan: {res['jawaban'][:120]}")
                if gagal >= maks_gagal_beruntun:
                    if verbose:
                        print("Gagal beruntun — kemungkinan kuota habis. Progres tersimpan; jalankan ulang nanti.")
                    return ck[ck["gen_tag"] == gtag]
                continue
            gagal = 0
            rec = {"kunci": kunci, "sistem": nama, "kueri_id": qid, "pertanyaan": g["pertanyaan"],
                   "jawaban": res["jawaban"], "konteks_str": res["konteks_str"], "gen_tag": gtag,
                   **hitung_legal_citation_f1(res["jawaban"], g["relevan"]),
                   "faithfulness": np.nan, "relevancy": np.nan, "alasan": "", "judge_tag": None,
                   "status_judge": "belum"}
            baru = pd.DataFrame([rec], columns=KOLOM_CKPT)
            ck = baru if ck.empty else pd.concat([ck, baru], ignore_index=True)
            simpan()
            if pakai_jeda:
                time.sleep(jeda)

    # Judge: nilai baris yang belum dinilai / gagal / dinilai judge lain
    if judge is not None:
        perlu = ck.index[(ck["gen_tag"] == gtag) & ((ck["status_judge"] != "ok") | (ck["judge_tag"] != judge_tag))]
        for i in perlu:
            r = ck.loc[i]
            f, rel, alasan, st = _nilai_judge(judge, r["pertanyaan"], r["konteks_str"], r["jawaban"])
            ck.loc[i, ["faithfulness", "relevancy", "alasan", "judge_tag", "status_judge"]] = [f, rel, alasan, judge_tag, st]
            simpan()
            if st == "gagal_api":
                print(f"Judge gagal (API) pada baris {i}: {alasan[:120]} — dihentikan, ulangi nanti.")
                break
            if "mock" not in str(judge_tag):
                time.sleep(jeda)
    return ck[ck["gen_tag"] == gtag]


if RUN_EVAL_GENERASI and LLM is not None and gold is not None and len(gold):
    df_gen = evaluasi_generasi(gold, {s: SISTEM_UTAMA[s] for s in SISTEM_EVAL_GENERASI}, k=5)
    kol = ["cit_p_ketat", "cit_r_ketat", "cit_f1_ketat", "cit_f1_longgar", "n_sitasi_tanpa_dok",
           "faithfulness", "relevancy"]
    print("\n=== RINGKASAN GENERASI (konfigurasi saat ini) ===")
    ring = df_gen.groupby("sistem")[kol].mean().round(4)
    ring["n_jawaban"] = df_gen.groupby("sistem").size()
    ring["n_judge_ok"] = df_gen[df_gen["status_judge"] == "ok"].groupby("sistem").size()
    print(ring.to_string())
    if df_gen["sistem"].nunique() == 2 and df_gen.groupby("sistem").size().min() >= 5:
        _d = df_gen.assign(**{"kueri_id": df_gen["kueri_id"].astype(int)})
        print(uji_wilcoxon(_d, [tuple(SISTEM_EVAL_GENERASI[::-1])], "cit_f1_ketat").to_string(index=False))
    ring.to_csv(HASIL_DIR / f"ringkasan_generasi_{TIMESTAMP}.csv", encoding="utf-8-sig")
else:
    print("Evaluasi generasi dilewati (cek RUN_EVAL_GENERASI, LLM, atau gold).")


=== RINGKASAN GENERASI (konfigurasi saat ini) ===
                 cit_p_ketat  cit_r_ketat  cit_f1_ketat  cit_f1_longgar  n_sitasi_tanpa_dok  faithfulness  relevancy  n_jawaban  n_judge_ok
sistem                                                                                                                                     
3. Hybrid (RRF)       0.4367       0.3417        0.3574          0.4120              0.5000        4.6739     4.5217         46          46
4. Graph-RAG          0.4304       0.3037        0.3383          0.3662              0.2174        4.8478     4.5217         46          46
                   Perbandingan       Metrik  Delta_Mean  n_beda    W+    W-  p_value    r_rb  p_holm  signifikan(α=.05,Holm)
4. Graph-RAG vs 3. Hybrid (RRF) cit_f1_ketat     -0.0191      28 184.5 221.5   0.6734 -0.0911  0.6734                   False


## §8 — Audit Akurasi Rujukan
- **§8a** (tanpa anotasi baru): membaca label lembar audit **v2** yang sudah Anda isi, lalu
  (1) presisi per `doc_id` → menguji dugaan A1, (2) memeriksa apakah ekstraksi v3 mengubah/membuang
  rujukan yang dilabel SALAH tanpa merusak yang BENAR.
- **§8b**: lembar audit **v3** berstrata. Label v2 otomatis disalin ke baris yang hasilnya identik
  (`catatan = prefill_v2`) sehingga anotasi baru hanya untuk rujukan yang berubah.

In [ ]:
LABEL_SET = ["BENAR", "SALAH_TARGET", "SALAH_JENIS", "BUKAN_RUJUKAN"]
KONTEKS_AUDIT = 160
LEMBAR_AUDIT_V2 = AUDIT_DIR / "lembar_audit_v2.xlsx"
LEMBAR_AUDIT_V3 = AUDIT_DIR / "lembar_audit_v3.xlsx"


def _wilson(kk: int, nn: int, z: float = 1.96):
    if nn == 0:
        return (float("nan"), float("nan"))
    p = kk / nn
    dd = 1 + z * z / nn
    c = (p + z * z / (2 * nn)) / dd
    h = z * math.sqrt(p * (1 - p) / nn + z * z / (4 * nn * nn)) / dd
    return (max(0.0, c - h), min(1.0, c + h))


def _kappa(a, b):
    a, b = list(a), list(b)
    nn = len(a)
    if nn == 0:
        return float("nan")
    po = sum(x == y for x, y in zip(a, b)) / nn
    pe = sum((a.count(k) / nn) * (b.count(k) / nn) for k in set(a) | set(b))
    return float("nan") if pe == 1 else (po - pe) / (1 - pe)


def _label_final(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    for c in ["label_A", "label_B", "label_adjudikasi"]:
        if c not in d:
            d[c] = ""
        d[c] = d[c].astype(object).where(d[c].notna(), "").astype(str).str.strip().str.upper()
    ok = (d["label_A"] != "") & ((d["label_B"] == "") | (d["label_A"] == d["label_B"]))
    final = d["label_A"].where(ok).astype(object)            # A (bila A=B atau B kosong)
    adj = d["label_adjudikasi"] != ""
    final[adj] = d.loc[adj, "label_adjudikasi"]               # adjudikasi selalu menang
    d["final"] = final
    return d


# ---------------------------------- §8a ----------------------------------
def diagnosis_audit_v2(path: Path = LEMBAR_AUDIT_V2):
    if not path.exists():
        print(f"§8a dilewati: {path.name} tidak ditemukan.")
        return None
    d = _label_final(pd.read_excel(path, sheet_name="internal_v2"))
    d = d[d["final"].notna()].copy()
    if d.empty:
        print("§8a dilewati: lembar v2 belum berlabel.")
        return None
    d["benar"] = (d["final"] == "BENAR").astype(int)
    d["omnibus"] = d["doc_id"].isin(DOK_OMNIBUS)

    print("=== §8a-1 Presisi rujukan internal (label v2) per dokumen — sampel, TIDAK dibobot strata ===")
    rows = []
    for did, x in d.groupby("doc_id"):
        lo, hi = _wilson(int(x["benar"].sum()), len(x))
        rows.append({"doc_id": did, "omnibus": did in DOK_OMNIBUS, "n": len(x),
                     "presisi": round(x["benar"].mean(), 3), "CI95": f"{lo:.2f}-{hi:.2f}",
                     "label_salah_terbanyak": x.loc[x["benar"] == 0, "final"].mode().iloc[0]
                     if (x["benar"] == 0).any() else "-"})
    per_doc = pd.DataFrame(rows).sort_values("n", ascending=False)
    print(per_doc.to_string(index=False))
    rendah = per_doc[(per_doc["n"] >= 5) & (per_doc["presisi"] < 0.5) & (~per_doc["omnibus"])]["doc_id"].tolist()
    if rendah:
        print(f"[SARAN] Presisi < 50% (n≥5) pada dokumen non-omnibus: {rendah} → periksa parser; "
              f"pertimbangkan KG_DOK_TANPA_RUJUKAN_INTERNAL = {set(rendah)}")
    print("\nPresisi sampel omnibus vs non-omnibus:")
    print(d.groupby("omnibus")["benar"].agg(n="size", presisi="mean").round(3).to_string())

    # §8a-2: apakah v3 mengubah baris yang dilabel salah?
    v3_hasil = ref_internal_v3.groupby("chunk_id")["hasil"].apply(set).to_dict()
    v3_kg = set(zip(ref_internal_v3.loc[mask_kg_internal(ref_internal_v3), "chunk_id"],
                    ref_internal_v3.loc[mask_kg_internal(ref_internal_v3), "hasil"]))

    def _status(r):
        if r["chunk_id"] not in POS:
            return "chunk_dibuang_(kembar)"
        if r["hasil_v2"] not in v3_hasil.get(r["chunk_id"], set()):
            return "target_diubah_v3"
        return "tetap_masuk_KG" if (r["chunk_id"], r["hasil_v2"]) in v3_kg else "tetap_tapi_keluar_KG"

    d["status_v3"] = d.apply(_status, axis=1)
    print("\n=== §8a-2 Label v2 × perlakuan ekstraksi v3 ===")
    print(pd.crosstab(d["final"], d["status_v3"], margins=True).to_string())
    tetap = d[d["status_v3"] == "tetap_masuk_KG"]
    if len(tetap):
        lo, hi = _wilson(int(tetap["benar"].sum()), len(tetap))
        print(f"\nPresisi sampel v2 yang TETAP masuk KG v3: {tetap['benar'].mean():.1%} "
              f"(n={len(tetap)}, Wilson {lo:.1%}–{hi:.1%}) — indikasi kasar, bukan estimasi berbobot.")
    rusak = d[(d["benar"] == 1) & (d["status_v3"] != "tetap_masuk_KG")]
    print(f"Rujukan BENAR (label v2) yang ikut berubah/keluar di v3: {len(rusak)} → periksa sebagai regresi.")
    per_doc.to_csv(HASIL_DIR / "audit_v2_per_dokumen.csv", index=False, encoding="utf-8-sig")
    d.to_csv(HASIL_DIR / "audit_v2_vs_v3.csv", index=False, encoding="utf-8-sig")
    return d


# ---------------------------------- §8b ----------------------------------
KUOTA_V3 = {"int_tunggal_valid": 70, "int_ekspansi_daftar_rentang": 50, "int_eks_bernomor": 40,
            "int_eks_tanpa_nomor": 20, "int_tak_ditemukan": 40, "int_amandemen": 40,
            "int_omnibus_tanpa_konteks": 30, "int_dari_hub": 30,
            "eks_dalam_korpus": 50, "eks_luar_korpus": 50, "eks_ambigu_jenis": 30}


def _konteks_audit(teks: str, s: int, e: int, w: int = KONTEKS_AUDIT) -> str:
    teks = str(teks)
    s = max(0, min(int(s), len(teks)))
    e = max(s, min(int(e), len(teks)))
    return teks[max(0, s - w):s] + "【" + teks[s:e] + "】" + teks[e:e + w]


def _stratifikasi(pop: pd.DataFrame, rng) -> pd.DataFrame:
    out = []
    for st, g in pop.groupby("strata"):
        n = KUOTA_V3.get(st, 50)
        samp = g.copy() if len(g) <= n else g.iloc[np.sort(rng.choice(len(g), n, replace=False))].copy()
        samp["N_strata"], samp["n_strata"] = len(g), len(samp)
        out.append(samp)
    return pd.concat(out, ignore_index=True) if out else pop


def populasi_internal_v3(ri: pd.DataFrame, rng) -> pd.DataFrame:
    teks_by = korpus.set_index("chunk_id")["isi"]
    masuk = mask_kg_internal(ri)
    rows = []
    for i, r in ri.iterrows():
        if r["sifat"] == "eksternal_bernomor":       st = "int_eks_bernomor"
        elif r["sifat"] == "eksternal_tanpa_nomor":  st = "int_eks_tanpa_nomor"
        elif r["sifat"] == "internal_amandemen":     st = "int_amandemen"
        elif r["omnibus_tanpa_konteks"]:             st = "int_omnibus_tanpa_konteks"
        elif r["dari_hub"]:                          st = "int_dari_hub"
        elif not r["valid"]:                         st = "int_tak_ditemukan"
        elif r["pola"] in ("daftar", "rentang"):     st = "int_ekspansi_daftar_rentang"
        else:                                        st = "int_tunggal_valid"
        rows.append({"strata": st, "doc_id": r["doc_id"], "chunk_id": r["chunk_id"], "pasal_asal": r["pasal_asal"],
                     "pola_v3": r["pola"], "sifat_v3": r["sifat"], "masuk_kg": bool(masuk.loc[i]),
                     "konteks": _konteks_audit(teks_by[r["chunk_id"]], r["ms"], r["me"]), "hasil_v3": r["hasil"]})
    return _stratifikasi(pd.DataFrame(rows), rng) if rows else pd.DataFrame()


def populasi_eksternal_v3(re_: pd.DataFrame, rng) -> pd.DataFrame:
    teks_by = korpus.set_index("chunk_id")["isi"]
    rows = []
    for _, r in re_[~re_["rujukan_diri"]].iterrows():
        st = "eks_dalam_korpus" if r["dalam_korpus"] else ("eks_ambigu_jenis" if r["ambigu_jenis"] else "eks_luar_korpus")
        doc_tuju = r["doc_tuju"] if pd.notna(r["doc_tuju"]) else "(luar_korpus)"
        rows.append({"strata": st, "doc_id": r["doc_id"], "chunk_id": r["chunk_id"], "kode_jenis": r["kode_jenis"],
                     "nomor_tahun": f"{r['nomor']}/{r['tahun']}", "masuk_kg": bool(r["dalam_korpus"]),
                     "konteks": _konteks_audit(teks_by[r["chunk_id"]], r["ms"], r["me"]),
                     "hasil_v3": f"{r['kode_jenis']} No.{r['nomor']} Th.{r['tahun']} -> {doc_tuju}"})
    return _stratifikasi(pd.DataFrame(rows), rng) if rows else pd.DataFrame()


def prefill_dari_v2(samp: pd.DataFrame, sheet_v2: str) -> pd.DataFrame:
    samp = samp.copy()
    for c in ["label_A", "label_B", "label_adjudikasi", "catatan"]:
        samp[c] = ""
    if not LEMBAR_AUDIT_V2.exists() or samp.empty:
        return samp
    try:
        v2 = pd.read_excel(LEMBAR_AUDIT_V2, sheet_name=sheet_v2)
    except Exception:
        return samp
    v2 = v2.drop_duplicates(["chunk_id", "hasil_v2"]).set_index(["chunk_id", "hasil_v2"])
    for i, r in samp.iterrows():
        key = (r["chunk_id"], r["hasil_v3"])
        if key in v2.index:
            lab = v2.loc[key]
            for c in ["label_A", "label_B", "label_adjudikasi"]:
                samp.at[i, c] = "" if pd.isna(lab.get(c)) else str(lab.get(c))
            samp.at[i, "catatan"] = "prefill_v2"
    return samp


def _tulis_sheet(ws, df_samp: pd.DataFrame, kol_meta: list, seed_offset: int = 0):
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.worksheet.datavalidation import DataValidation
    kol_anotasi = ["label_A", "label_B", "label_adjudikasi", "catatan"]
    kol = ["id"] + kol_meta + kol_anotasi + ["N_strata", "n_strata"]
    ws.append(kol)
    for c in ws[1]:
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", start_color="1F4E78")
    d = df_samp.sample(frac=1, random_state=RANDOM_SEED + seed_offset).reset_index(drop=True)
    for i, (_, r) in enumerate(d.iterrows(), 1):
        ws.append([i if k == "id" else (r[k] if k in d.columns and pd.notna(r[k]) else "") for k in kol])
    fill_in, fill_pre = PatternFill("solid", start_color="FFF2CC"), PatternFill("solid", start_color="E2EFDA")
    for row in ws.iter_rows(min_row=2):
        pre = row[kol.index("catatan")].value == "prefill_v2"
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
            if kol[c.column - 1] in kol_anotasi:
                c.fill = fill_pre if pre else fill_in
    for k in ["label_A", "label_B", "label_adjudikasi"]:
        dv = DataValidation(type="list", formula1='"' + ",".join(LABEL_SET) + '"', allow_blank=True)
        ws.add_data_validation(dv)
        col = ws.cell(1, kol.index(k) + 1).column_letter
        dv.add(f"{col}2:{col}{max(ws.max_row, 2)}")
    lebar = {"id": 6, "strata": 26, "doc_id": 20, "chunk_id": 28, "konteks": 65, "hasil_v3": 40, "catatan": 14}
    for i, n in enumerate(kol, 1):
        ws.column_dimensions[ws.cell(1, i).column_letter].width = lebar.get(n, 16)
    ws.freeze_panes = "A2"


def evaluasi_sheet(path: Path, nama_sheet: str, B: int = BOOTSTRAP_B):
    d = _label_final(pd.read_excel(path, sheet_name=nama_sheet))
    if not (d["label_A"] != "").any():
        print(f"[{nama_sheet}] belum dianotasi.")
        return None
    s = d[d["final"].notna()].copy()
    s["benar"] = (s["final"] == "BENAR").astype(int)
    rng = np.random.default_rng(RANDOM_SEED)

    def _estimasi(sub):
        g = {st: x["benar"].to_numpy(float) for st, x in sub.groupby("strata")}
        N = {st: float(sub.loc[sub["strata"] == st, "N_strata"].iloc[0]) for st in g}
        tot = sum(N.values())
        est = lambda smp: sum(N[st] / tot * smp[st].mean() for st in smp)  # noqa: E731
        boot = [est({st: rng.choice(v, len(v), replace=True) for st, v in g.items()}) for _ in range(B)]
        return est(g), np.percentile(boot, 2.5), np.percentile(boot, 97.5)

    tbl = []
    for st, x in s.groupby("strata"):
        lo, hi = _wilson(int(x["benar"].sum()), len(x))
        tbl.append({"strata": st, "N": int(x["N_strata"].iloc[0]), "n": len(x),
                    "prefill_v2": int((x["catatan"].astype(str) == "prefill_v2").sum()),
                    "precision": round(x["benar"].mean(), 3), "CI95_Wilson": f"{lo:.3f}-{hi:.3f}"})
    print(f"\n=== HASIL AUDIT v3 ({nama_sheet}) ===")
    print(pd.DataFrame(tbl).to_string(index=False))
    e, lo, hi = _estimasi(s)
    print(f"Precision berbobot strata (seluruh): {e:.1%}  CI95 bootstrap [{lo:.1%} - {hi:.1%}]")
    s_kg = s[s["masuk_kg"].astype(str).str.upper().isin(["TRUE", "1"])]
    if len(s_kg):
        e, lo, hi = _estimasi(s_kg)
        print(f"Precision berbobot strata (masuk KG): {e:.1%}  CI95 bootstrap [{lo:.1%} - {hi:.1%}]  ← angka RQ1")
    dua = d[(d["label_A"] != "") & (d["label_B"] != "")]
    if len(dua):
        kp = _kappa(dua['label_A'], dua['label_B'])
        print(f"Cohen's kappa A-B: {kp:.3f} ({len(dua)} baris)" if not math.isnan(kp)
              else f"Cohen's kappa A-B: tak terdefinisi (semua label sama; {len(dua)} baris, kesepakatan "
                   f"{(dua['label_A'] == dua['label_B']).mean():.0%})")
    return pd.DataFrame(tbl)


if RUN_AUDIT:
    audit_v2_vs_v3 = diagnosis_audit_v2()
    if LEMBAR_AUDIT_V3.exists():
        print(f"\nLembar audit v3 sudah ada (tidak ditimpa): {LEMBAR_AUDIT_V3}")
    else:
        from openpyxl import Workbook
        rng = np.random.default_rng(RANDOM_SEED)
        s_int = prefill_dari_v2(populasi_internal_v3(ref_internal_v3, rng), "internal_v2")
        s_eks = prefill_dari_v2(populasi_eksternal_v3(ref_eksternal_v3, rng), "eksternal_v2")
        wb = Workbook()
        ws = wb.active; ws.title = "internal_v3"
        _tulis_sheet(ws, s_int, ["strata", "doc_id", "chunk_id", "pasal_asal", "pola_v3", "sifat_v3",
                                 "masuk_kg", "konteks", "hasil_v3"], 0)
        _tulis_sheet(wb.create_sheet("eksternal_v3"), s_eks,
                     ["strata", "doc_id", "chunk_id", "kode_jenis", "nomor_tahun", "masuk_kg", "konteks", "hasil_v3"], 100)
        wb.save(LEMBAR_AUDIT_V3)
        n_pre = int((s_int["catatan"] == "prefill_v2").sum() + (s_eks["catatan"] == "prefill_v2").sum())
        print(f"\nLembar audit v3 dibuat: {LEMBAR_AUDIT_V3}")
        print(f"  internal_v3: {len(s_int)} sampel | eksternal_v3: {len(s_eks)} sampel | "
              f"{n_pre} baris sudah terisi dari label v2 (hijau). Isi baris kuning lalu jalankan ulang.")
    for sh in ["internal_v3", "eksternal_v3"]:
        try:
            evaluasi_sheet(LEMBAR_AUDIT_V3, sh)
        except Exception as e:
            print(f"Evaluasi {sh} dilewati: {e}")

=== §8a-1 Presisi rujukan internal (label v2) per dokumen — sampel, TIDAK dibobot strata ===
           doc_id  omnibus  n  presisi      CI95 label_salah_terbanyak
    Perppu_2_2022     True 67    0.060 0.02-0.14           SALAH_JENIS
        UU_6_2023     True 58    0.121 0.06-0.23         BUKAN_RUJUKAN
Perdirjen_07_2021    False 41    0.854 0.72-0.93           SALAH_JENIS
       PP_28_2025    False 22    0.727 0.52-0.87         BUKAN_RUJUKAN
       PP_46_2021    False 13    0.692 0.42-0.87          SALAH_TARGET
    Permen_5_2021    False  6    1.000 0.61-1.00                     -
       UU_12_2011    False  3    1.000 0.44-1.00                     -
       UU_36_1999    False  2    1.000 0.34-1.00                     -
       UU_32_2002    False  2    0.500 0.09-0.91         BUKAN_RUJUKAN
       PP_52_2000    False  1    1.000 0.21-1.00                     -
       PP_49_2005    False  1    1.000 0.21-1.00                     -
 Perdirjen_3_2024    False  1    0.000 0.00-0.79       

## §9 — Self-test (uji unit + invariant pada korpus Anda)
Semua uji harus ✓. Uji yang gagal ditandai ✗ beserta pesan; sel ini memunculkan error di akhir bila ada yang gagal.

In [ ]:
def jalankan_selftest():
    hasil = []

    def uji(nama):
        def dek(fn):
            try:
                fn(); hasil.append((nama, True, ""))
            except Exception as e:  # noqa: BLE001
                hasil.append((nama, False, f"{type(e).__name__}: {e}"))
            return fn
        return dek

    @uji("Wilcoxon: tanda effect size mengikuti arah selisih (B1)")
    def _():
        det = pd.DataFrame([{"kueri_id": i, "sistem": s, "nDCG@10": v}
                            for i, (a, b) in enumerate(zip([.9, .8, .7, .6, .9, .8, .7, .5], [.1, .2, .3, .4, .2, .3, .9, .4]))
                            for s, v in (("A", a), ("B", b))])
        r = uji_wilcoxon(det, [("A", "B"), ("B", "A")])
        assert r.loc[0, "r_rb"] > 0 and r.loc[1, "r_rb"] < 0, r
        assert abs(r.loc[0, "p_value"] - r.loc[1, "p_value"]) < 1e-9

    @uji("Holm: p terkoreksi ≥ p mentah dan monoton")
    def _():
        p = np.array([0.01, 0.04, 0.03, 0.2])
        a = holm(p)
        assert (a >= p).all() and np.allclose(a[np.argsort(p)], np.maximum.accumulate(a[np.argsort(p)]))

    @uji("Ekstraksi sitasi jawaban: dokumen+pasal, kedua urutan (B4)")
    def _():
        s = ekstrak_sitasi("Menurut [PP Nomor 46 Tahun 2021 — Pasal 64 ayat (1)] dan Pasal 33 "
                           "Undang-Undang Nomor 36 Tahun 1999, serta Pasal 5.")
        assert (("PP", 46, 2021), "64") in s, s
        assert (("UU", 36, 1999), "33") in s, s
        assert any(p == "5" for _, p in s), s

    @uji("Citation F1 ketat menolak pasal dari dokumen salah; longgar menerima")
    def _():
        m = hitung_legal_citation_f1("Lihat [PP Nomor 46 Tahun 2021 — Pasal 1].", "UU_36_1999__P1__n2")
        assert m["cit_f1_ketat"] == 0.0 and m["cit_f1_longgar"] == 1.0, m
        m = hitung_legal_citation_f1("Lihat [UU Nomor 36 Tahun 1999 — Pasal 1].", "UU_36_1999__P1__n2")
        assert m["cit_f1_ketat"] == 1.0, m

    @uji("Citation F1: dokumen kembar dianggap setara")
    def _():
        if not _KANON:
            return
        (a, b), = list(DOK_KEMBAR.items())[:1]
        ka = parse_docid(a)
        jenis = {"PERPPU": "Perppu", "UU": "UU", "PP": "PP"}.get(ka[0], ka[0])
        m = hitung_legal_citation_f1(f"[{jenis} Nomor {ka[1]} Tahun {ka[2]} — Pasal 7]", f"{b}__P7")
        assert m["cit_f1_ketat"] == 1.0, m

    @uji("Ekstraksi v2: daftar, rentang, eksternal bernomor, intra-pasal")
    def _():
        mini = pd.DataFrame({
            "doc_id": ["PP_9_2020"] * 6 + ["UU_8_2010"],
            "pasal": ["3", "5", "6", "7", "8", "9", "4"], "ayat": [pd.NA] * 7,
            "isi": ["Pasal 3 ayat (2) berlaku sebagaimana dimaksud dalam Pasal 5 dan Pasal 6.",
                    "Ketentuan Pasal 6 sampai dengan Pasal 8 berlaku.",
                    "Merujuk Pasal 4 Undang-Undang Nomor 8 Tahun 2010.", "x", "y", "z", "w"],
            "chunk_id": [f"c{i}" for i in range(7)]})
        mini["pasal"] = mini["pasal"].astype("string")
        ri, _ = ekstrak_rujukan_v2(mini)
        r0 = ri[ri.chunk_id == "c0"]
        assert set(r0.pasal_tuju) == {"5", "6"} and (r0.pola == "daftar").all(), r0   # Pasal 3 diri sendiri dilewati
        assert set(ri[ri.chunk_id == "c1"].pasal_tuju) == {"6", "7", "8"}
        r2 = ri[ri.chunk_id == "c2"].iloc[0]
        assert r2.sifat == "eksternal_bernomor" and r2.doc_tuju == "UU_8_2010" and r2.valid

    @uji("Ekstraksi v3: rujukan dalam blok amandemen diarahkan ke UU yang diubah (A1)")
    def _():
        mini = pd.DataFrame({
            "doc_id": ["OMNI_1_2023"] * 4 + ["UU_8_2010"] * 2,
            "bab": ["VI", "VI", "VI", "VII", "I", "I"],
            "pasal": ["71", "71", "71", "80", "33", "34"], "ayat": [pd.NA] * 6,
            "isi": ["Beberapa ketentuan dalam Undang-Undang Nomor 8 Tahun 2010 tentang X diubah sebagai berikut:",
                    "Ketentuan Pasal 33 diubah sehingga berbunyi sebagaimana dimaksud dalam Pasal 34.",
                    "lanjutan Pasal 34.", "Pasal 71 berlaku sejak diundangkan.", "a", "b"],
            "chunk_id": [f"o{i}" for i in range(6)]})
        mini["pasal"] = mini["pasal"].astype("string")
        ktx, _ = deteksi_konteks_amandemen(mini, {"OMNI_1_2023"})
        assert ktx["o1"] == ("UU", 8, 2010) and ktx["o3"] is None, ktx
        ri, _ = ekstrak_rujukan_v2(mini, ktx, set())
        r1 = ri[ri.chunk_id == "o1"]
        assert (r1.sifat == "internal_amandemen").all() and set(r1.doc_tuju) == {"UU_8_2010"}, r1
        assert r1.valid.all()
        r3 = ri[ri.chunk_id == "o3"]
        assert r3.empty or r3.omnibus_tanpa_konteks.all() or (r3.pasal_tuju == "71").sum() == 0

    @uji("KG: tidak ada sisi merujuk dari rujukan omnibus-tanpa-konteks / pasal-hub")
    def _():
        if EKSTRAKSI_AKTIF != "v3":
            return
        assert not ref_kg_internal["omnibus_tanpa_konteks"].any()
        if KG_ABAIKAN_RUJUKAN_DARI_HUB:
            assert not ref_kg_internal["dari_hub"].any()
        for u, v, d in KG.edges(data=True):
            assert "tipe" in KG.nodes[u] and "tipe" in KG.nodes[v]

    @uji("Gold: semua ID relevan ada di korpus setelah remap")
    def _():
        if gold is None:
            return
        ids = {x for s in gold["relevan"] for x in str(s).split(";")}
        assert ids <= set(POS), list(ids - set(POS))[:5]

    @uji("Top-k stabil terhadap k (prasyarat evaluasi pool) untuk semua sistem")
    def _():
        qs = list(gold["pertanyaan"][:3]) if gold is not None and len(gold) else [_q_demo]
        for nama, f in SISTEM_UTAMA.items():
            for q in qs:
                a, b = f(q, 10)["chunk_id"].tolist(), f(q, K_POOL)["chunk_id"].tolist()[:10]
                assert a == b, f"{nama}: {q[:40]}"

    @uji("Ekspansi graf memilih saudara dengan kemiripan tertinggi (A4)")
    def _():
        q = gold["pertanyaan"].iloc[0] if gold is not None and len(gold) else _q_demo
        qv = encode_query(q)
        cid = next(c for c, p in PASAL_DARI_SATUAN.items() if len(SATUAN_PER_PASAL[p]) > KUOTA_SISI["memuat"] + 2)
        sk, rel = perluas_graf_v3([cid], {cid: 1.0}, qv, aktif_sisi={"memuat"})
        sib = [c for c in SATUAN_PER_PASAL[PASAL_DARI_SATUAN[cid]] if c != cid]
        sims = {c: max(0.0, float(embeddings[POS[c]] @ qv)) for c in sib}
        dipilih = set(sk)
        n_pos = sum(v > 0 for v in sims.values())
        assert len(dipilih) == min(KUOTA_SISI["memuat"], n_pos), (len(dipilih), n_pos)
        lain = [sims[c] for c in sib if c not in dipilih]
        if dipilih and lain:
            assert min(sims[c] for c in dipilih) >= max(lain) - 1e-6

    @uji("Ablasi: setiap jenis sisi yang punya kandidat benar-benar bisa menyumbang")
    def _():
        q = gold["pertanyaan"].iloc[0] if gold is not None and len(gold) else _q_demo
        seeds = retrieve_hybrid(q, k=K_SEED)["chunk_id"].tolist()
        sk, rel = perluas_graf_v3(seeds, {c: 1.0 for c in seeds}, encode_query(q))
        jk = jangkauan_per_relasi(seeds)
        qv = encode_query(q)
        for r in SEMUA_SISI:
            if KUOTA_SISI.get(r, 0) > 0 and any(float(embeddings[POS[c]] @ qv) > 0 for c in jk[r]):
                assert any(c in jk[r] for c in sk), f"sisi {r} tidak pernah menyumbang kandidat"

    @uji("Filter metadata dihormati Graph-RAG (termasuk hasil ekspansi)")
    def _():
        did = korpus["doc_id"].value_counts().index[0]
        h = retrieve_graphrag(_q_demo, 10, filter_meta={"doc_id": did})
        assert (h["chunk_id"].map(lambda c: korpus.at[POS[c], "doc_id"]) == did).all()

    @uji("Label asal: item 'graf' tidak berasal dari pool hybrid")
    def _():
        h = retrieve_graphrag(_q_demo, 10)
        pool = set(retrieve_hybrid(_q_demo, k=max(10, N_POOL_HYBRID))["chunk_id"])
        assert not (set(h.loc[h.asal == "graf", "chunk_id"]) & pool)
        assert set(h.loc[h.asal != "graf", "chunk_id"]) <= pool

    @uji("Metrik pasal: dedup dari pool sebelum potong k (B9)")
    def _():
        g = pd.DataFrame({"relevan": ["D__P2__a1"]}, index=[0])
        pred = ["D__P1__a1", "D__P1__a2", "D__P1__a3", "D__P2__a1"]
        m = hitung_metrik({("X", 0): (pred, 0.0)}, g, "pasal", k_list=(2,))
        assert m.loc[0, "Recall@2"] == 1.0 and m.loc[0, "MRR"] == 0.5

    @uji("Kunci cache embedding berubah bila teks/model berubah (B6)")
    def _():
        a = kunci_embedding("m", "", ["x", "y"])
        assert a != kunci_embedding("m", "", ["x", "z"]) and a != kunci_embedding("m2", "", ["x", "y"])
        assert a == kunci_embedding("m", "", ["x", "y"])

    @uji("Checkpoint: jawaban gagal tidak disimpan; resume tidak memanggil ulang; judge diulang (B2/B3)")
    def _():
        import tempfile
        tmp = Path(tempfile.mkdtemp()) / "ck.csv"
        gsub = pd.DataFrame({"pertanyaan": ["q satu", "q dua"], "relevan": [gold["relevan"].iloc[0] if gold is not None else "x"] * 2})
        ret = lambda q, k: retrieve_bm25(_q_demo, k)  # noqa: E731

        def gagal(p): raise RuntimeError("429 kuota")
        out = evaluasi_generasi(gsub, {"S": ret}, llm=gagal, llm_tag="t-fail", judge=None, judge_tag=None,
                                ckpt_path=tmp, jeda=0, verbose=False)
        assert len(out) == 0
        panggil = {"n": 0}

        def ok(p):
            panggil["n"] += 1; return _llm_mock(p)
        out = evaluasi_generasi(gsub, {"S": ret}, llm=ok, llm_tag="mock-ok", judge=None, judge_tag=None, ckpt_path=tmp, jeda=0)
        assert len(out) == 2 and panggil["n"] == 2 and (out.status_judge == "belum").all()
        out = evaluasi_generasi(gsub, {"S": ret}, llm=ok, llm_tag="mock-ok", judge=_llm_mock, judge_tag="mock-j", ckpt_path=tmp, jeda=0)
        assert panggil["n"] == 2 and (out.status_judge == "ok").all() and out.faithfulness.notna().all()

    @uji("Intent deontik terdeteksi")
    def _():
        assert "SANKSI" in deteksi_intent("Apa sanksi bagi pengguna frekuensi tanpa izin?")
        assert "KEWAJIBAN" in deteksi_intent("Apa kewajiban pemegang ISR?")
        assert not deteksi_intent("Bagaimana prosedur perpanjangan ISR?")

    for nama, ok, pesan in hasil:
        print(("✓ " if ok else "✗ ") + nama + ("" if ok else f"\n    → {pesan}"))
    n_gagal = sum(not ok for _, ok, _ in hasil)
    print(f"\n{len(hasil) - n_gagal}/{len(hasil)} uji lolos.")
    if n_gagal:
        raise AssertionError(f"{n_gagal} self-test gagal")


if RUN_SELFTEST:
    jalankan_selftest()

✓ Wilcoxon: tanda effect size mengikuti arah selisih (B1)
✓ Holm: p terkoreksi ≥ p mentah dan monoton
✓ Ekstraksi sitasi jawaban: dokumen+pasal, kedua urutan (B4)
✓ Citation F1 ketat menolak pasal dari dokumen salah; longgar menerima
✓ Citation F1: dokumen kembar dianggap setara
✓ Ekstraksi v2: daftar, rentang, eksternal bernomor, intra-pasal
✓ Ekstraksi v3: rujukan dalam blok amandemen diarahkan ke UU yang diubah (A1)
✓ KG: tidak ada sisi merujuk dari rujukan omnibus-tanpa-konteks / pasal-hub
✓ Gold: semua ID relevan ada di korpus setelah remap
✓ Top-k stabil terhadap k (prasyarat evaluasi pool) untuk semua sistem
✓ Ekspansi graf memilih saudara dengan kemiripan tertinggi (A4)
✓ Ablasi: setiap jenis sisi yang punya kandidat benar-benar bisa menyumbang
✓ Filter metadata dihormati Graph-RAG (termasuk hasil ekspansi)
✓ Label asal: item 'graf' tidak berasal dari pool hybrid
✓ Metrik pasal: dedup dari pool sebelum potong k (B9)
✓ Kunci cache embedding berubah bila teks/model berubah (B6)
✓